# Always Run

## Imports, Physical Channels, Qubit Definitions, Functions and More

TODO:
 -  **Have a saving figure and data procedure that is never redundant and automatic based on the cells being run (import from elsewhere)
 - Have a resonator tracking and saving procedure that then is assigned to the qubit object
 - (Eventually) Would be nice to have a central matplotlib figure template with everything nicely configured and customized (for fun)
 - Xi measurement

-QAOut (QACHANNELS/0/OUTPUT) - C6

-QAIn (QACHANNELS/0/INPUT)   - B4

-AWG1 (SGCHANNELS/0/OUTPUT)  - A7 - Nothing                                 

-AWG2 (SGCHANNELS/1/OUTPUT)  - A16 - F1 Fast Flux Drive

-AWG3 (SGCHANNELS/2/OUTPUT)  - A5 - SC2PHI Charge Drive

-AWG4 (SGCHANNELS/3/OUTPUT)  - C3 - F0 Drive

-AWG5 (SGCHANNELS/4/OUTPUT)  - A1 - F1 Charge Drive

-AWG6 (SGCHANNELS/5/OUTPUT)  - A14 - SC2PHI Fast Flux Drive

-C2PHI_Flux: DC4

-F0_Flux_Pulse: A16

-HGKP_Flux_Pulse: A14

(Driving all Transmons through F1's drive since it is closest on the chip)

In [ ]:
from pathlib import Path
import datetime
from datetime import date
import pandas as pd
import time
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import sys
import logging
import math as m
import dill
from scipy.optimize import curve_fit
import statistics as stat
from typing import Callable
from qcodes.instrument_drivers.yokogawa.GS200 import GS200

import laboneq
from laboneq.simple import *
import laboneq.pulse_sheet_viewer.pulse_sheet_viewer as psv
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

from laboneq.analysis.fitting import (
    lorentzian,
    oscillatory,
    oscillatory_decay,
    exponential_decay,
)

In [ ]:
# Just be careful about the drive lines.
descriptor_shfqc = """
instruments:
  SHFQC:
  - address: DEV12195
    uid: device_shfqc
    interface: 1gbe
    options: SHFQC/PLUS/QC6CH

  # HDAWG:
  # - address: DEV8371
  #   uid: device_hdawg
  #   interface: 1gbe
  #   options: HDAWG8/SKW
  #   reference_clock_source: external_clock_signal

connections: 
  device_shfqc:
    - acquire_signal: T0/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: T0/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: T0/drive_line
      ports: SGCHANNELS/4/OUTPUT

    - acquire_signal: T1/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: T1/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: T1/drive_line
      ports: SGCHANNELS/4/OUTPUT

    - acquire_signal: F0/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: F0/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: F0/drive_line
      ports: SGCHANNELS/3/OUTPUT
    # - iq_signal: F0/flux_drive_line
      # ports: SGCHANNELS/3/OUTPUT

    - acquire_signal: F1/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: F1/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: F1/drive_line
      ports: SGCHANNELS/4/OUTPUT
    - iq_signal: F1/flux_line
      ports: SGCHANNELS/1/OUTPUT

    - acquire_signal: LC2PHI/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: LC2PHI/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: LC2PHI/drive_line
      ports: SGCHANNELS/2/OUTPUT
    - iq_signal: LC2PHI/flux_drive_line
      ports: SGCHANNELS/5/OUTPUT

    - acquire_signal: MC2PHI/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: MC2PHI/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: MC2PHI/drive_line
      ports: SGCHANNELS/5/OUTPUT

    - acquire_signal: HC2PHI/acquire_line
      ports: [QACHANNELS/0/INPUT]
    - iq_signal: HC2PHI/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - iq_signal: HC2PHI/drive_line
      ports: SGCHANNELS/0/OUTPUT
    # - iq_signal: HC2PHI/flux_drive_line
      # ports: SGCHANNELS/1/OUTPUT

  # device_hdawg:
  #   - iq_signal: F1/flux_line
  #     ports: [SIGOUTS/0, SIGOUTS/1]
"""
# the very last signal (iq_signal for flux_drive_line) needs to manually
# configure a LO (even if it will only be used in LF mode)

The logical signal group is what defines the set of logical parameters for measurements. This needs to get assigned to an experimental signal group in the Experiment class connecting theory with device_setup (shown later).

In [ ]:
# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor_shfqc, # yaml fully describes logical signal layout
    server_host="localhost",  # ip address of the LabOne dataserver
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="my_setup",  # setup name
)
# device_setup.instrument_by_uid('device_hdawg').reference_clock_source='external'

In [ ]:
# define shortcut to logical signals for convenience
lsg = {
    qubit_name: device_setup.logical_signal_groups[qubit_name].logical_signals
    for qubit_name in device_setup.logical_signal_groups.keys()}

## Qubit Parameters

In [ ]:
# cos2phi_v2_separated_style measurement
T0 = Transmon.from_logical_signal_group(
    uid='T0',
    lsg=device_setup.logical_signal_groups['T0'],
    parameters=TransmonParameters(
        readout_resonator_frequency=6.399e9,
        readout_lo_frequency=6.4e9,
        resonance_frequency_ge=4e9,
        drive_lo_frequency=4e9,
        readout_integration_delay=88e-9,
        readout_range_out=-20,
        drive_range=10,
        readout_range_in=0,
        user_defined={
            'pulse_length': 50e-9,
            'readout_len': 2.048e-6, 
            'time_domain_reset_length': 20e-9,
            'cw_reset_length': 5e-9,
            'readout_amp': 1,
            'amplitude_pi': None,
        }
    )
)

T1 = Transmon.from_logical_signal_group(
    uid='T1',
    lsg=device_setup.logical_signal_groups['T1'],
    parameters=TransmonParameters(
        readout_resonator_frequency=6.565e9,
        readout_lo_frequency=6.4e9,
        resonance_frequency_ge=3.565e9,
        drive_lo_frequency=3.6e9,
        readout_integration_delay=88e-9,
        readout_range_out=-5,
        drive_range=0,
        readout_range_in=10,
        user_defined={
            'pulse_length': 500e-9,
            'readout_len': 2e-6,
            'time_domain_reset_length': 300e-6,
            'cw_reset_length': 5e-9,
            'readout_amp': 0.5,
            'amplitude_pi': 0.32,
            'amplitude_pi/2': 0.18,
        }
    )
)

F0 = Transmon.from_logical_signal_group(
    uid='F0',
    lsg=device_setup.logical_signal_groups['F0'],
    parameters=TransmonParameters(
        readout_resonator_frequency=6.739e9,
        readout_lo_frequency=6.8e9,
        resonance_frequency_ge=6.315e9,
        drive_lo_frequency=6.4e9,
        readout_integration_delay=88e-9, 
        readout_range_out=-10,
        drive_range=10,
        readout_range_in=10, #5,
        user_defined={
            'pulse_length': 1000e-9,
            'readout_len': 2e-6, 
            'time_domain_reset_length': 100e-6,
            'cw_reset_length': 100e-9,
            'readout_amp': 1,
            'amplitude_pi': 1,
            'amplitude_pi/2': 0.5,
            'current_sweetspot': None,
            'current_setpoint': None,
        }
    )
)

F1 = Transmon.from_logical_signal_group(
    uid='F1',
    lsg=device_setup.logical_signal_groups['F1'],
    parameters=TransmonParameters(
        readout_resonator_frequency=6866759709, #6.8708e9,
        readout_lo_frequency=6.8e9,
        resonance_frequency_ge=386e6,
        drive_lo_frequency=3.6e9, 
        readout_integration_delay=88e-9, 
        readout_range_out=0, #-20,
        drive_range=10,
        readout_range_in=10, #5, 
        user_defined={
            'pulse_length': 2000e-9,
            'readout_len': 2e-6, 
            'time_domain_reset_length': 200-6,
            'cw_reset_length': 5e-9,
            'readout_amp': 0.8,
            'amplitude_pi': 0.18,
            'amplitude_pi/2': 0.36,
            'current_sweetspot': None,
            'current_setpoint': None,
        }
    )
)
'''
LC2PHI = Transmon.from_logical_signal_group(
    uid='LC2PHI',
    lsg=device_setup.logical_signal_groups['LC2PHI'],
    parameters=TransmonParameters(
        readout_resonator_frequency=7.0975e9,
        readout_lo_frequency=7e9,
        resonance_frequency_ge=5.35e9,
        drive_lo_frequency=5.2e9,
        readout_integration_delay=88e-9,
        readout_range_out=-20,
        drive_range=0,
        readout_range_in=5,
        user_defined={
            'pulse_length': 4000e-9,
            'readout_len': 2e-6, 
            'time_domain_reset_length': 5e-6,
            'cw_reset_length': 500e-9,
            'readout_amp': 1,
            'amplitude_pi': None,
            'current_sweetspot': None,
            'current_setpoint': -25e-6,
        }
    )
)

MC2PHI = Transmon.from_logical_signal_group(
   uid='MC2PHI',
    lsg=device_setup.logical_signal_groups['MC2PHI'],
    parameters=TransmonParameters(
        readout_resonator_frequency=7.27e9,
        readout_lo_frequency=7.2e9,
        resonance_frequency_ge=4e9, 
        drive_lo_frequency=4e9, 
        readout_integration_delay=88e-9, 
        readout_range_out=-10,
        drive_range=0,
        readout_range_in=5,
        user_defined={
            'pulse_length': 50e-9,
            'readout_len': 2e-6,
            'time_domain_reset_length': 200e-9,
            'cw_reset_length': 5e-9,
            'readout_amp': None,
            'amplitude_pi': None,
            'current_sweetspot': None,
            'current_setpoint': None,
        }
    )
)

HC2PHI = Transmon.from_logical_signal_group(
    uid='HC2PHI',
    lsg=device_setup.logical_signal_groups['HC2PHI'],
    parameters=TransmonParameters(
        readout_resonator_frequency=7.5e9,
        readout_lo_frequency=7.4e9,
        resonance_frequency_ge=4e9, 
        drive_lo_frequency=4e9,
        readout_integration_delay=88e-9,
        readout_range_out=-25,
        drive_range=0,
        readout_range_in=5,
        user_defined={
            'pulse_length': 500e-9,
            'readout_len': 2e-6,
            'time_domain_reset_length': 200e-9,
            'cw_reset_length': 5e-9,
            'readout_amp': 0.8,
            'amplitude_pi': 1,
            'amplitude_pi/2': 1,
            'current_sweetspot': None,
            'current_setpoint': None,
        }
    )
)
''';
# QuantumElement

In [ ]:
qubit=F1 #Defines which qubit we are actually measuring
averages=2**10
device_setup.set_calibration(qubit.calibration()) #Uses qubit params for calib

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.user_defined['readout_len'],
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

try: qubit.res_to_current = res_to_current
except: print('No res_to_current defined during this runtime session')

In [ ]:
device_setup

In [ ]:
# Thought I was cooking. Seems like the open connections with the ZI box makes
# the current session unpicklible. Perhaps there can be some context switching
# shinnanagans which makes this ok. Will have to see...
# '''
with open(str(datetime.date.today()) + 'ipython_session.pkl', 'wb') as f:
    dill.dump_session(f, byref=False)
# '''

## Defining Functions
Defining functions for plotting statistics and other useful functions

In [ ]:
def exp_decay(x, a, b, c):
    """
    General exponential decay function: f(x) = a * exp(-b * x) + c
    
    Parameters:
    -----------
    x : array-like
        Independent variable values
    a : float
        Amplitude (scaling factor)
    b : float
        Decay rate (higher values mean faster decay)
    c : float
        Vertical offset (asymptotic value as x approaches infinity)
    
    Returns:
    --------
    array-like
        Function values at points x
    """
    return a * np.exp(-b * x) + c

def compare_binary_files(file_path_1, file_path_2):
    """
    Compares the binary content of two files.

    Args:
        file_path_1: Path to the first file.
        file_path_2: Path to the second file.

    Returns:
        True if the files are identical, False otherwise.
    """
    try:
        with open(file_path_1, 'rb') as file1, open(file_path_2, 'rb') as file2:
            while True:
                chunk1 = file1.read(4096)
                chunk2 = file2.read(4096)
                if chunk1 != chunk2:
                    return False
                if not chunk1:
                    return True
    except FileNotFoundError:
        return False

def plot_box_whisker_time_series(
        datetime_array: np.ndarray, float_array: np.ndarray,
        t_type_str: str, time_interval_str: str ='2min'):
    '''Plots data using box and whisker according to set time interval'''
    data = pd.DataFrame({'time': atetime_array, 'value': float_array})
    data.set_index('time', inplace=True)
    datantime = data.resample(time_interval_str).apply(list) 
    datantime.dropna(inplace=True)
    time_labels = datantime.index
    float_data_per_interval = datantime['value']
    plt.figure(figsize=(16, 6))
    plt.boxplot(
        float_data_per_interval, 
        positions=range(len(time_labels)), 
        widths=0.7)
    plt.xticks(
        ticks=range(len(time_labels)), 
        labels=time_labels.strftime('%H:%M'), 
        rotation=45)
    plt.xlabel('Time (Intervals)')
    plt.ylabel(f'{t_type_str} (us)')
    plt.title(f'{t_type_str} Times over Time')
    plt.tight_layout()
    plt.show()

def measure_data_stats(T_run: Callable, n_runs: int):
    '''Outputs statistics on data gathered'''
    global datetime_array #global so if the func stalls, it will still be there
    global fit_out #global so if the func stalls, it will still be there
    fit_out = np.empty((2, n_runs))
    datetime_array = np.empty((n_runs), dtype=datetime.datetime)
    for i in range(n_runs):
        fits = T_run()
        datetime_array[i] = datetime.datetime.now()
        fit_out[:,i] = fits 
    return [datetime_array, fit_out]

def stat_out(data: np.ndarray, t_type_str: str):
    mean = np.nanmean(data)
    std = np.nanstd(data)
    print(t_type_str, ' = ', str(mean), '+/-', str(std))
    return [mean, std]

def adjust_phase(
        IQ_data: np.ndarray, frequency: np.ndarray, 
        electrical_delay: float,
        ) -> np.ndarray:
    adjusted_complex = np.exp(1j*electrical_delay*2*np.pi*frequency)*IQ_data
    flattened_angle = np.unwrap(np.angle(adjusted_complex))
    flattened_angle = flattened_angle - np.mean(flattened_angle)
    return flattened_angle 

def data_directory_update():
    '''Updates data directory to today's date. Returns str for path'''
    date = datetime.date.today()
    datadir = Path('data\\' + str(date))
    if not os.path.exists(datadir):
        os.makedirs(datadir)
    return str(datadir) + '\\'
    
datadir = data_directory_update() # might as well instantly update for notebook

def non_redund_save_fig(fig, name):
    '''A function to prevent figure overwrite issues'''
    datadir = data_directory_update()
    i = 1
    while True:
        fig_name = Path(str(datadir) + f'/{name}_{i}.png')
        if os.path.isfile(fig_name) is False:
            fig_name = Path(str(datadir) + f'/{name}_{i}')
            fig.savefig(fig_name)
            break
        else:
            i = i+1

def non_redund_save_pd(pd_data, name):
    '''A function to prevent pd overwrite issues'''
    datadir = data_directory_update()
    i = 1
    while True:
        pd_name = Path(str(datadir) + f'/{name}_{i}.csv')
        if os.path.isfile(pd_name) is False:
            pd_name = Path(str(datadir) + f'/{name}_{i}.csv')
            pd_data.to_csv(pd_name)
            break
        else:
            i += 1

def non_redund_save_csv(csv_data, name):
    '''A function to prevent csv overwrite issues'''
    datadir = data_directory_update()
    i = 1
    while True:
        csv_name = Path(str(datadir) + f'/{name}_{i}.csv')
        if os.path.isfile(csv_name) is False:
            csv_name = Path(str(datadir) + f'/{name}_{i}.csv')
            csv_data.to_csv(csv_name)
            break
        else:
            i += 1
            
def create_arbitrary_pulse(fname: str, uid: str) -> laboneq.dsl.experiment.pulse.PulseSampled:
    '''Takes in a .csv file and outputs a pulse that has the same shape as the csv file sample-wise (typ. 0.5ns).
    
    This should be a two column array from -1 to 1 for each column, specifying the pulse shape.
    First column is for I values, second is for Q. Note Q intensity on pulse sheet will have additional negative sign.
    '''
    array = np.loadtxt(fname, delimeter=',')
    pulse = pulse_library.sampled_pulse(samples=array, uid=uid)
    return pulse           



# have a script that identifies the peak for a given transition after a flux pulse
def approximate_inverse(x_values, y_values, target_y):
    """
    Approximates the inverse function f^-1(y) at a specific y value
    using linear interpolation on a discrete, monotonically increasing function.
    
    Parameters:
    x_values (array-like): The x coordinates of the function f(x)
    y_values (array-like): The y coordinates of the function f(x), assumed to be monotonically increasing
    target_y (float): The y value for which to find the approximate x = f^-1(y)
    
    Returns:
    float: The approximate x value corresponding to target_y
    str: A message indicating the result status
    """
    # Convert inputs to numpy arrays for easier handling
    x = np.array(x_values)
    y = np.array(y_values)
    
    # Check if the function is monotonically increasing
    if not np.all(np.diff(y) >= 0):
        return None, "Error: Input function is not monotonically increasing"
    
    # Check if target_y is within the range of y_values
    if target_y < y[0] or target_y > y[-1]:
        return None, f"Error: Target y={target_y} is outside the range of provided y values [{y[0]}, {y[-1]}]"
    
    # Find the indices where target_y would be inserted to maintain order
    idx = np.searchsorted(y, target_y)
    
    # If target_y exactly matches a y value, return the corresponding x value
    if idx < len(y) and y[idx] == target_y:
        return x[idx], f"Exact match found at x={x[idx]}"
    
    # Otherwise, perform linear interpolation between the two closest points
    idx_lower = idx - 1
    
    # Calculate the interpolated x value
    x_lower, y_lower = x[idx_lower], y[idx_lower]
    x_upper, y_upper = x[idx], y[idx]
    
    # Linear interpolation formula: x = x_lower + (target_y - y_lower) * (x_upper - x_lower) / (y_upper - y_lower)
    x_interp = x_lower + (target_y - y_lower) * (x_upper - x_lower) / (y_upper - y_lower)
    
    return x_interp

In [ ]:
'''
Real time calls: Seems as if the QCoDes intruments cannot be pickled into
the experiment when it is sent to the HFSQC. To get around this, do not pass
the QCoDes object as a parameter to the call. Just have that name be in the 
global namespace which makes sure it is handled by the computer.
'''
def change_dc_current(session, new_current, step_time):
    dc.ramp_current(new_current, 10e-6, step_time)
    print("DC Current " + str(new_current))

def change_coil_current(session, new_current, step_time):
    coil.ramp_current(new_current, 1e-6, step_time)
    print("Current coil current: " + str(new_current))
    
def sweep_progress(session, linearsweepparameter):
    print(str(linearsweepparameter.count))



# Yoko Connection
(Sometimes tempramental, so don't always run)

In [ ]:
# try:
#     dc = GS200('yoko_right', address = 'TCPIP0::192.168.4.157::inst0::INSTR',)
# except Exception as e:
#     print(e)

try:
    coil = GS200('yoko_left', address = 'TCPIP0::192.168.4.208::inst0::INSTR',)
except Exception as e:
    print(e)

In [ ]:
# dc.BNC_out.set('ready') # sets low signal signature after move is completed
# dc.BNC_out.get()
# dc.ramp_current(0e-3, 1e-6,0.02)
# dc.output('off')
# dc.source_mode('CURR')
# dc.ramp_current(208e-6, 1e-6, 0.02)
# dc.output('on')
# dc.current_range(0.01)

# coil.BNC_out.set('ready') # sets low signal signature after move is completed
# coil.BNC_out.get()

# coil.output('off')
# coil.source_mode('CURR')
coil.ramp_current(-119e-6, 1e-6, 0.02)
# coil.ramp_current(0e-3, 1e-6,0.02)
# coil.output('on')
# coil.current_range(0.01)

# dc.current.get()
coil.current.get()

# Readout Delay Calibration

## Experiment

In [ ]:
readout_delay = LinearSweepParameter(
    uid='readout_delay',
    start=75e-9,
    stop=125e-9,
    count=51)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line'])]
    
exp = Experiment(
    uid="Readout_Delay_Calibration",
    signals=exp_signals,)

exp_calibration = Calibration()
with exp.sweep(uid="readout_delay", parameter=readout_delay,):
    exp_calibration['acquire'] = SignalCalibration(port_delay=readout_delay)
    exp.set_calibration(exp_calibration)
    with exp.acquire_loop_rt(
        uid="shots",
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        # readout pulse and data acquisition
        with exp.section(uid="readout",):
            exp.play(
                signal="measure",
                pulse=readout_pulse)
            exp.acquire(
                signal='acquire',
                handle='single_freq_data',
                length=qubit.parameters.user_defined['readout_len'],
                kernel=readout_pulse # What is actually integrated against
            )
        with exp.section(uid="delay_between_readout", length=210e-9):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect(use_async_api=True);
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
device_setup.logical_signal_groups['F1'].get_calibration()

## Plot and Analyze Data

In [ ]:
time_delay = my_acquired_results.axis[0]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
# phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

fig, ax = plt.subplots(2, 1, figsize=(8,6))
ax[0].plot(time_delay*1e9, amplitude)
ax[0].set_title(f'{qubit.uid} Sweeping Acquire Delay')
ax[0].set_xlabel('Time delay (ns)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].vlines(87, np.min(amplitude), np.max(amplitude), 'r')
ax[0].grid()

ax[1].plot(time_delay*1e9, phase)
ax[1].set_title(f'{qubit.uid} Sweeping Acquire Delay')
ax[1].set_xlabel('Time delay (ns)')
ax[1].set_ylabel('Phase')
ax[1].vlines(87, np.min(phase), np.max(phase))
ax[1].grid()
fig.tight_layout()

# Global Resonator Trace

## Experiment

In [ ]:
def non_redund_name(name) -> str:
    '''A function to prevent overwrite issues'''
    new_data_dir = data_directory_update()
    temp_name = name
    index = 2
    while os.path.isfile(new_data_dir + temp_name) is True:
        temp_name = name + str(index)
        index += 1
    return new_data_dir + temp_name

In [ ]:
#--- Experiment Name ---
exp_name = qubit.uid + "GlobalResTrace"

#--- Defining Parameter Sweeps ---
readout_lo_freq_sweep = LinearSweepParameter(
    uid='Readout_LO',
    start=6.6e9,
    stop=6.6e9,
    count=1)
readout_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency',
    start=100e6,
    stop=200e6,
    count=401)

#--- Signal Mappings ---
exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),]

#--- Defining Sections and Sweeps and Experiment ---
exp = Experiment(
    uid='Global Resonator Trace',
    signals=exp_signals,)
RO_LO_Sweep = Sweep(
    uid='Readout LO Frequency Sweep',
    parameters=readout_lo_freq_sweep)
RT_Loop = AcquireLoopRt(
    uid='Shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)
AWG_Freq_Sweep = Sweep(
    uid='Readout Frequency Sweep',
    parameters=readout_freq_sweep,
    reset_oscillator_phase=False)
Meas_Acquire = Section(uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='single_freq_data', 
    length=qubit.parameters.user_defined['readout_len'])
Delay_After_Count = Section(uid='Delay Between Readout')

#--- Properly Defining Nesting Order ---
exp.add(RO_LO_Sweep)
RO_LO_Sweep.add(RT_Loop)
RT_Loop.add(AWG_Freq_Sweep)
AWG_Freq_Sweep.add(Meas_Acquire)
AWG_Freq_Sweep.add(Delay_After_Count)
Delay_After_Count.reserve(signal='measure')
Delay_After_Count.reserve(signal='acquire')

#--- Calibrating Experiment ---
readout_osc = Oscillator(
    "readout_osc",
    frequency=readout_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
readout_lo = Oscillator(
    "readout_lo",
    frequency=readout_lo_freq_sweep,
    modulation_type=ModulationType.HARDWARE)

exp_calibration = Calibration()
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp.set_calibration(exp_calibration)

#--- Compiling Session ---
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

#--- Running Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results

# Extracts data from the exp.acquire method with the same key name
my_acquired_results = my_results.acquired_results['single_freq_data']

## Plot and Analyze Data

In [ ]:
# Extracting useful data
lo_array = my_acquired_results.axis[0]
AWG_freqs = my_acquired_results.axis[1]
freqs = np.empty((0))
for lo in lo_array:
    freqs = np.append(freqs, lo + AWG_freqs,)
IQ_data = my_acquired_results.data.ravel()
amplitude = np.abs(IQ_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

# Plot
fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title('Wide Range Pulsed Trace')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[1].plot(freqs, phase)
ax[1].set_title('Wide Range Pulsed Trace')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].set_ylabel('Phase')
fig.tight_layout()

In [ ]:
session.save_results(str(data_directory_update()) + '/' + qubit.uid + '2d_flux_sweep_upper_right')

# Local Resonator Trace
Actually just works to put in the frequency values for start and stop.
The LO will adjust as necessary on the backend. Nice.

## Experiment

In [ ]:
try: 
    qubit.parameters.readout_resonator_frequency = res_to_current(coil.current())
    device_setup.set_calibration(qubit.calibration())
    print('At current set readout freq')
except Exception as e:
    print('Failed to set readout freq')
    print(e)

exp_name = qubit.uid + "LocalResTrace"

ro_frequency_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency - 15e6,
    stop=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency - 5e6

, 
    count=201)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),]
    

exp = Experiment(
    uid="Resonator Trace",
    signals=exp_signals,)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,
):
    with exp.sweep(
        uid='Readout Frequency Sweep',
        parameter=ro_frequency_sweep,
        reset_oscillator_phase=False
    ):
        with exp.section(uid="drive"):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                    amplitude=0.18), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
            )
        with exp.section(uid="Pulsed Single Frequency Readout", play_after='drive'):
            exp.play(
                signal="measure",
                pulse=readout_pulse
            )
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=3e-6
            )
        with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['cw_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_frequency_sweep,
    modulation_type=ModulationType.HARDWARE,)
drive_osc = Oscillator(
    "drive_osc",
    frequency=30e6)
drive_lo_lf = Oscillator(
    "drive_lo",
    frequency=3.6e9)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_lo_lf)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

# For plotting current vs single resonator point
freqs = my_acquired_results.axis[0] + qubit.parameters.readout_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

# Plots
fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].plot(freqs, phase)
ax[1].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
fig.tight_layout()
# ax[0].vlines(qubit.res_to_current(coil.current()),np.min(amplitude), np.max(amplitude), color='r')
# ax[0].vlines(qubit.parameters.readout_resonator_frequency, np.min(amplitude), np.max(amplitude), colors='r');
# ax[1].vlines(qubit.parameters.readout_resonator_frequency, np.min(phase), np.max(phase), colors='r');

In [ ]:
try: 
    qubit.parameters.readout_resonator_frequency = res_to_current(coil.current()) + 5e6
    device_setup.set_calibration(qubit.calibration())
    print('At current set readout freq')
except Exception as e:
    print('Failed to set readout freq')
    print(e)

exp_name = qubit.uid + "LocalResTrace"

ro_frequency_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency + 10e6,
    stop=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency - 10e6, 
    count=201)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line'])]

exp = Experiment(
    uid="Resonator Trace",
    signals=exp_signals,)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,
):
    with exp.sweep(
        uid='Readout Frequency Sweep',
        parameter=ro_frequency_sweep,
        reset_oscillator_phase=False
    ):
        with exp.section(uid="Pulsed Single Frequency Readout"):
            exp.play(
                signal="measure",
                pulse=readout_pulse
            )
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=3e-6
            )
        with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['cw_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_frequency_sweep,
    modulation_type=ModulationType.HARDWARE,)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

## Plot and Analyze Data

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

# For plotting current vs single resonator point
freqs = my_acquired_results.axis[0] + qubit.parameters.readout_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

# Plots
fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].plot(freqs, phase)
ax[1].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
fig.tight_layout()
# ax[0].vlines(qubit.res_to_current(coil.current()),np.min(amplitude), np.max(amplitude), color='r')
# ax[0].vlines(qubit.parameters.readout_resonator_frequency, np.min(amplitude), np.max(amplitude), colors='r');
# ax[1].vlines(qubit.parameters.readout_resonator_frequency, np.min(phase), np.max(phase), colors='r');

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
def arctan_fit(freqs, omega, phs_offset, offset):
    '''Rough phase fitting after normalizing avg phase amplitude to zero'''
    return 2*np.arctan(omega*(freqs-phs_offset)) + offset
(popt, b) = curve_fit(arctan_fit, freqs, phase, p0=[-1e-9, 6.8675e9, 1]);
plt.plot(freqs, arctan_fit(freqs, *popt))
plt.plot(freqs, phase)
print(popt[1])

# Punchout
This works, but the power sweep is NEAR_TIME. Putting it in the REAL_TIME section makes the play channel seem like there are multiple envelopes for the readout pulse? Might be becasue of some timing or alignement issue... For now it seems fine given that the loop only needs to be executed a handful of times.

## Experiment

In [ ]:
exp_name = qubit.uid + "Punchout"

#Note this is a power sweep of the pulse amplitude, not of powers! The conversion from amplitude to powers is squared.
readout_pulse_punchout = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.user_defined['readout_len'],
    amplitude=1,
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

ro_frequency_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_frequency-5e6,
    stop=qubit.parameters.readout_frequency+5e6,
    count=201) 

power_sweep = SweepParameter(
    uid='Readout_Power',
    values=np.logspace(start=-1.5,stop=0,num=15,base=10))

exp = Experiment(
    uid="Punchout",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ]
)

with exp.sweep(
    uid='Power sweep',
    parameter=power_sweep,
):
    with exp.acquire_loop_rt(
        uid='shots',
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.SPECTROSCOPY,
    ):
    
        with exp.sweep(
            uid='Readout_Frequency_Sweep',
            parameter=ro_frequency_sweep,
            reset_oscillator_phase=False
        ):
            with exp.section(uid="single_res_point_readout"):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse_punchout,
                    amplitude=power_sweep,
                )
                exp.acquire(
                    signal='acquire',
                    handle='single_freq_data',
                    length=4e-6
                )
            with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['cw_reset_length']):
                exp.reserve(signal="measure")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_frequency_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
# For plotting punchout
freqs = my_acquired_results.axis[1]+qubit.parameters.readout_lo_frequency
IQ_data = my_acquired_results.data
normalized_amp_data = np.divide(np.abs(IQ_data).T, np.mean(np.abs(IQ_data), axis=1))
normalized_dB_data = np.log10(normalized_amp_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

amplitude = my_acquired_results.axis[0]
power = 10*np.log10(amplitude**2)+qubit.parameters.readout_range_out

graphing_data = np.abs(phase.T)# normalized_dB_data #phase.T

fig, ax = plt.subplots(1,2, figsize=(9,6))
cmap0 = ax[0].pcolor(amplitude,
             freqs,
             graphing_data,
             shading='nearest')
ax[0].set_title(f'{qubit.uid} Punchout of Resonator (Pulse Amplitude)')
ax[0].set_xlabel('Pulse Amplitude')
ax[0].set_ylabel('Readout Frequency (GHz)')
ax[0].set_xscale('log')
cmap1 = ax[1].pcolor(power,
             freqs,
             graphing_data,
             shading='nearest',
             vmin=-5,
             vmax=5)
fig.colorbar(cmap0, ax=ax[0])
fig.colorbar(cmap1, ax=ax[1])
ax[1].set_title(f'{qubit.uid} Punchout of Resonator (dBm)')
ax[1].set_xlabel('dBm')
ax[1].set_ylabel('Readout Frequency (GHz)')
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

# Resonator 1D Flux Sweep

## Experiment

In [ ]:
exp_name = qubit.uid + "1DFluxSweep"

ro_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_frequency-2e6,
    stop=qubit.parameters.readout_frequency+3e6,
    count=201)

dc_sweep_param = LinearSweepParameter(
    uid='DC Current',
    start=(175.7-(212.7/2))*1e-6,
    stop=(175.7+(212.7/2))*1e-6,
    count=213)

coil_sweep_param = LinearSweepParameter(
    start=-130*1e-6,
    stop=-100*1e-6,
    count=31)

current_sweep_param = coil_sweep_param # SETS WHICH DC SOURCE DOES THE SWEEPING

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line'])]

exp = Experiment(
    uid="1D Current vs Resonator",
    signals=exp_signals,)

Current_Sweep = Sweep(
    uid=current_sweep_param.uid,
    parameters=current_sweep_param,
    execution_type=ExecutionType.NEAR_TIME)
Current_Sweep.call(
    change_coil_current,
    new_current=current_sweep_param,
    step_time=0.01)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)

RO_Freq_Sweep = Sweep(
    uid=ro_freq_sweep.uid,
    parameters=ro_freq_sweep,
    execution_type=ExecutionType.REAL_TIME,)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'])

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['cw_reset_length'],
    play_after=Meas_Acquire)

exp.add(Current_Sweep)
Current_Sweep.add(RT_Loop)
RT_Loop.add(RO_Freq_Sweep)
RO_Freq_Sweep.add(Meas_Acquire)
RO_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()

readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect();
session.register_neartime_callback(change_coil_current)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

### TODO
Save the frequency sweep data relations

In [ ]:
# session.save_results(str(data_directory_update()) + '/' + qubit.uid + 'res_tracked')
# session = Session.load(str(data_directory_update()) + '/' + qubit.uid + 'res_tracked')

## Plot and Analyze Data

In [ ]:
# For plotting 1D flux sweep
freqs = my_acquired_results.axis[1] + qubit.parameters.readout_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)
# currents = my_acquired_results.axis[0][0]*1e6
currents = my_acquired_results.axis[0]*1e6

fig, ax = plt.subplots(1,2, figsize=(15,6))
cmap0 = ax[0].pcolor(currents,
    freqs,
    amplitude.T,
    shading='nearest')
ax[0].set_title(f'{qubit.uid} Resonator Current Response')
ax[0].set_xlabel('Currents (uA)')
ax[0].set_ylabel('Readout Frequency (GHz)')
cmap1 = ax[1].pcolor(currents,
    freqs,
    phase.T,
    shading='nearest',)
fig.colorbar(cmap0, ax=ax[0])
fig.colorbar(cmap1, ax=ax[1])
ax[1].set_title(f'{qubit.uid} Resonator Current Response')
ax[1].set_xlabel('Currents (uA)')
ax[1].set_ylabel('Readout Frequency (GHz)')
fig.tight_layout()

In [ ]:
print(device_setup)

In [ ]:
# Tracks the resonator doing a simple arctan fit on the phase
tracked_resonator = np.empty(amplitude.shape[0])
ro_res_freq = qubit.parameters.readout_resonator_frequency

def arctan_fit(freqs, omega, phs_offset, offset):
    '''Rough phase fitting after normalizing avg phase amplitude to zero'''
    return 2*np.arctan(omega*(freqs-phs_offset)) + offset

def make_ro_freq():
    '''A closure for new_ro_values which is to be assigned to a qubit'''
    old_currents = my_acquired_results.axis[0]
    old_values = tracked_resonator
    def res_to_current(new_currents: float|np.ndarray):
        new_ro_values = np.interp(new_currents, old_currents, old_values)
        return new_ro_values
    return res_to_current

for i, phs in enumerate(phase):
    try:
        (popt, b) = curve_fit(arctan_fit, freqs, phs, p0=[-1e-9, 6.8675e9, 1])
        opt_freq = popt[1]
        tracked_resonator[i] = opt_freq
    except Exception as e:
        tracked_resonator[i] = tracked_resonator[i-1]
        print(e)
plt.scatter(currents, tracked_resonator)
# plt.ylim(6.855e9, 6.88e9)
# plt.xlim(-150, -50)
# plt.grid()

res_to_current = make_ro_freq()

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
# Tracks the resonator by doing a lorentzian fit on the amplitude
tracked_resonator = np.empty(amplitude.shape[0])
ro_res_freq = qubit.parameters.readout_resonator_frequency

def make_ro_freq():
    '''A closure for new_ro_values which is to be assigned to a qubit'''
    old_currents = my_acquired_results.axis[0]
    old_values = tracked_resonator
    def res_to_current(new_currents: float|np.ndarray):
        new_ro_values = np.interp(new_currents, old_currents, old_values)
        return new_ro_values
    return res_to_current

for i, amp in enumerate(amplitude):
    try:
        (popt, b) = lorentzian.fit(freqs, amp, 1000e3, ro_res_freq, -1e7, 1);
        opt_freq = popt[1]
        # print(f"Resonant frequency: {opt_freq} GHz")
        tracked_resonator[i] = opt_freq
    except Exception as e:
        tracked_resonator[i] = tracked_resonator[i-1]
        print(e)
plt.scatter(currents, tracked_resonator)
plt.ylim(6.86e9, 6.87e9)

res_to_current = make_ro_freq()

# Resonator 1D Flux Sweep (Relational)

In [ ]:
exp_name = qubit.uid + "1DFluxSweepRelational"

ro_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_frequency-6e6,
    stop=qubit.parameters.readout_frequency+6e6,
    count=121)

dc_sweep_param = LinearSweepParameter(
    uid='DC Current',
    start=(175.7-(212.7/2))*1e-6,
    stop=(175.7+(212.7/2))*1e-6,
    count=213)

coil_sweep_param = LinearSweepParameter(
    uid='Coil Current',
    start=(63.6-(163.8/2))*1e-6,
    stop=(63.6+(163.8/2))*1e-6,
    count=213)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line'])]

exp = Experiment(
    uid="1D Current vs Resonator",
    signals=exp_signals,)

Current_Sweep = Sweep(
    uid=dc_sweep_param.uid,
    parameters=[dc_sweep_param, coil_sweep_param], 
    execution_type=ExecutionType.NEAR_TIME)
Current_Sweep.call(
    change_dc_current,
    new_current=dc_sweep_param,
    step_time=0.01)
Current_Sweep.call(
    change_coil_current,
    new_current=coil_sweep_param,
    step_time=0.01)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)

RO_Freq_Sweep = Sweep(
    uid=ro_freq_sweep.uid,
    parameters=ro_freq_sweep,
    execution_type=ExecutionType.REAL_TIME,)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'])

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['cw_reset_length'],
    play_after=Meas_Acquire)

exp.add(Current_Sweep)
Current_Sweep.add(RT_Loop)
RT_Loop.add(RO_Freq_Sweep)
RO_Freq_Sweep.add(Meas_Acquire)
RO_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()

readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
session.register_neartime_callback(change_coil_current)
session.register_neartime_callback(change_dc_current)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
# Tracks the resonator doing a simple arctan fit on the phase
tracked_resonator = np.empty(amplitude.shape[0])
ro_res_freq = qubit.parameters.readout_resonator_frequency

def arctan_fit(freqs, omega, phs_offset, offset):
    '''Rough phase fitting after normalizing avg phase amplitude to zero'''
    return 2*np.arctan(omega*(freqs-phs_offset)) + offset

def make_ro_freq():
    '''A closure for new_ro_values which is to be assigned to a qubit'''
    old_currents = my_acquired_results.axis[0][0]
    old_values = tracked_resonator
    def res_to_current(new_currents: float|np.ndarray):
        new_ro_values = np.interp(new_currents, old_currents, old_values)
        return new_ro_values
    return res_to_current

for i, phs in enumerate(phase):
    try:
        (popt, b) = curve_fit(arctan_fit, freqs, phs, p0=[-1e-9, 6.8675e9, 1])
        opt_freq = popt[1]
        tracked_resonator[i] = opt_freq
    except Exception as e:
        tracked_resonator[i] = tracked_resonator[i-1]
        print(e)
plt.scatter(currents, tracked_resonator)
# plt.ylim(6.855e9, 6.88e9)
# plt.xlim(-150, -50)
# plt.grid()

res_to_current = make_ro_freq()

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
# Tracks the resonator by doing a lorentzian fit on the amplitude
tracked_resonator = np.empty(amplitude.shape[0])
ro_res_freq = qubit.parameters.readout_resonator_frequency

def make_ro_freq():
    '''A closure for new_ro_values which is to be assigned to a qubit'''
    old_currents = my_acquired_results.axis[0][0]
    old_values = tracked_resonator
    def res_to_current(new_currents: float|np.ndarray):
        new_ro_values = np.interp(new_currents, old_currents, old_values)
        return new_ro_values
    return res_to_current

for i, amp in enumerate(amplitude):
    try:
        (popt, b) = lorentzian.fit(freqs, amp, 1000e3, ro_res_freq, -1e7, 1);
        opt_freq = popt[1]
        # print(f"Resonant frequency: {opt_freq} GHz")
        tracked_resonator[i] = opt_freq
    except Exception as e:
        tracked_resonator[i] = tracked_resonator[i-1]
        print(e)
plt.scatter(currents, tracked_resonator)
# plt.ylim(6.86e9, 6.87e9)

res_to_current = make_ro_freq()

# Resonator 2D Flux Sweep
- Would be nice to have a faster flux sweep functionality at the end (or beginning?) of each loop

## Experiment

In [ ]:
exp_name = qubit.uid + "2DFluxSweep"

dc_sweep_param = LinearSweepParameter(
    uid='DC_Current',
    start=-300e-6,
    stop=600e-6,
    count=351)

coil_sweep_param = LinearSweepParameter(
    uid='Coil_Current',
    start=-100e-6,
    stop=150e-6,
    count=351)

exp_signal = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),]

exp = Experiment(
    uid="2D Current Sweep",
    signals=exp_signal)

Coil_Sweep = Sweep(
    uid=coil_sweep_param.uid,
    parameters=coil_sweep_param,
    execution_type=ExecutionType.NEAR_TIME)
Coil_Sweep.call(
    change_coil_current,
    new_current=coil_sweep_param,
    step_time=0.01)

DC_Sweep = Sweep(
    uid=dc_sweep_param.uid,
    parameters=dc_sweep_param,
    execution_type=ExecutionType.NEAR_TIME)
DC_Sweep.call(
    change_dc_current,
    new_current=dc_sweep_param,
    step_time=0.01)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    acquisition_type=AcquisitionType.SPECTROSCOPY_IQ,
    averaging_mode=AveragingMode.CYCLIC)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'],)
    
Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['cw_reset_length'],
    play_after=Meas_Acquire)

Outer_Sweep = Coil_Sweep
Inner_Sweep = DC_Sweep

exp.add(Outer_Sweep)
Outer_Sweep.add(Inner_Sweep)
Inner_Sweep.add(RT_Loop)
RT_Loop.add(Meas_Acquire)
RT_Loop.add(Delay_After_Count)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
session.register_neartime_callback(change_dc_current,)
session.register_neartime_callback(change_coil_current,)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
#plotting resonator over flux
amp_data = np.abs(my_acquired_results.data)
phase_data = np.angle(my_acquired_results.data)
db_data = np.log10(amp_data)

coil_currents = my_acquired_results.axis[0]
dc_currents = my_acquired_results.axis[1]

fig, ax = plt.subplots(1,2, figsize=(15,6))
cmap0 = ax[0].pcolor(coil_currents*1e6,
             dc_currents*1e6,
             db_data.T,
             shading='nearest')
ax[0].set_title(f'{qubit.uid} Resonator 2D Flux Response')
ax[0].set_xlabel('Coil Currents (uA)')
ax[0].set_ylabel('On-Chip DC (uA)')
cmap1 = ax[1].pcolor(coil_currents*1e6,
             dc_currents*1e6,
             phase_data.T,
             shading='nearest')
ax[1].set_title(f'{qubit.uid} Resonator 2D Flux Response')
ax[1].set_xlabel('Coil Currents (uA)')
ax[1].set_ylabel('On-Chip DC (uA)')
fig.colorbar(cmap0, ax=ax[0])
fig.colorbar(cmap1, ax=ax[1])
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
# session.save_results(str(data_directory_update()) + '/' + qubit.uid + '_fine_large')
# fig.savefig(str(data_directory_update()) + '/c2phi_fine_large.png')
# np.savetxt(str(data_directory_update()) + '/c2phi_fine_large', my_acquired_results.data)

In [ ]:
outer_sweep_param = my_acquired_results.axis[0]


inner_sweep_param = my_acquired_results.axis[1]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
db_amp = np.log10(amplitude)
phase = np.angle(IQ_data)

fig, ax = plt.subplots(1,2, figsize=(15,6))
cmap0 = ax[0].pcolor(inner_sweep_param*1e6,
    outer_sweep_param*1e6,
    amplitude,
    shading='nearest')
ax[0].set_title(f'{qubit.uid} Resonator Current Response')
ax[0].set_xlabel(f'{Inner_Sweep.uid} (uA)')
ax[0].set_ylabel(f'{Outer_Sweep.uid} (uA)')
cmap1 = ax[1].pcolor(inner_sweep_param*1e6,
    outer_sweep_param*1e6,
    phase,
    shading='nearest',)
cbar0 = fig.colorbar(cmap0, ax=ax[0],)
cbar1 = fig.colorbar(cmap1, ax=ax[1])
cbar0.set_label('Measured log10(Amplitude)')
cbar1.set_label('Phase')
ax[1].set_title(f'{qubit.uid} Resonator Current Response')
ax[1].set_xlabel(f'{Inner_Sweep.uid} (uA)')
ax[1].set_ylabel(f'{Outer_Sweep.uid} (uA)')
fig.tight_layout()

# Global Qubit Spectroscopy
-Unclear still how to have the LO naturally move during the sweep. Maybe build it into a function call?
-For some reason, the drive frequency sweep is upset that it is in real-time. Need to investigate further.

## Experiment

In [ ]:
exp_name = qubit.uid + "GlobalSpec"

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.user_defined['readout_len'],
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

drive_pulse = pulse_library.gaussian_square(
    uid=f'drive_pulse_{qubit.uid}',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=1,
    sigma=0.2)

drive_lo_freq_sweep = LinearSweepParameter(
    uid='Drive_LO',
    start=1e9,
    stop=8e9,
    count=8)

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=-500e6,
    stop=498e6,
    count=201)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])]

exp = Experiment(
    uid="Spectroscopy",
    signals=exp_signals)

Drive_LO_Freq_Sweep = Sweep(
    uid=drive_lo_freq_sweep.uid,
    parameters=drive_lo_freq_sweep,
    execution_type=ExecutionType.NEAR_TIME)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,)

Drive_Freq_Sweep = Sweep(
    uid=drive_freq_sweep.uid,
    parameters=drive_freq_sweep,
    execution_type=ExecutionType.REAL_TIME)

Drive = Section(uid = 'Drive')
Drive.play(
    signal='drive',
    pulse=drive_pulse,)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',
    play_after=Drive)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'],
    kernel=readout_pulse) #this is important to actually integrate against so the result is non-zero

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['time_domain_reset_length'],
    play_after=Meas_Acquire)

exp.add(Drive_LO_Freq_Sweep)
Drive_LO_Freq_Sweep.add(RT_Loop)
RT_Loop.add(Drive_Freq_Sweep)
Drive_Freq_Sweep.add(Drive)
Drive_Freq_Sweep.add(Meas_Acquire)
Drive_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
drive_lo = Oscillator(
    'drive_lo',
    frequency=drive_lo_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE,)

exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_lo)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)

exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=100,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key nam

## Plot and Analyze Data

In [ ]:
drive_lo_array = my_acquired_results.axis[0]
drive_AWG_freqs = my_acquired_results.axis[1]
drive_freqs = np.empty((0))
for drive_lo in drive_lo_array:
    drive_freqs = np.append(drive_freqs, drive_lo + drive_AWG_freqs,)

In [ ]:
# For plotting current vs single resonator point
IQ_data = my_acquired_results.data.ravel()
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))

fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(drive_freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Spectrum')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude')
# ax[0].set_ylim(1, 5)
ax[0].grid()
ax[1].plot(drive_freqs, phase)
ax[1].set_title(f'{qubit.uid} Spectrum')
ax[1].set_xlabel('Drive Frequency (GHz)')
ax[1].set_ylabel('Phase')
# ax[1].set_ylim(2, 2.7)
ax[1].grid()
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

# Local Qubit Spectroscopy LF

In [ ]:
exp_name = qubit.uid + "LocalSpecLF"

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=qubit.parameters.resonance_frequency_ge-500e6,
    stop=qubit.parameters.resonance_frequency_ge+500e6,
    count=201
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Drive Frequency Sweep',
        parameter=drive_freq_sweep,
    ):
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=qubit.parameters.user_defined["pulse_length"],
                    amplitude=1), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
            )
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.reserve('drive')
            exp.play(
                signal="measure",
                pulse=readout_pulse,)
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=2e-6,
                kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout",
                         play_after="single_res_point_readout",
                         length=10e-6): #qubit.parameters.user_defined['cw_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")
            exp.reserve(signal="drive")


exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency= (res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE,)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=0)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.LF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=100,)

In [ ]:
(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency - 15e4),

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
# For plotting current vs single resonator point
freqs = my_acquired_results.axis[0]
# freqs = my_acquired_results.axis[0] + qubit.parameters.drive_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase - np.mean(phase)

fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Spectrum near Transition')
ax[0].set_xlabel('Drive Frequency (Hz)')
ax[0].set_ylabel('Amplitude')
ax[0].grid()
# ax[0].vlines(qubit.parameters.drive_frequency_ge+qubit.parameters.drive_lo_frequency, np.min(amplitude), np.max(amplitude), colors='r');
ax[1].plot(freqs, phase)
ax[1].set_title(f'{qubit.uid} Spectrum near Transition')
ax[1].set_xlabel('Drive Frequency (Hz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
ax[0].vlines(qubit.parameters.resonance_frequency_ge, np.min(amplitude), np.max(amplitude), colors='r');
ax[1].vlines(qubit.parameters.resonance_frequency_ge, np.min(phase), np.max(phase), colors='r')
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

# Local Qubit Spectroscopy RF

## Experiment

In [ ]:
# try: 
#     qubit.parameters.readout_resonator_frequency = res_to_current(coil.current())
#     device_setup.set_calibration(qubit.calibration())
#     print('At current set readout freq')
# except Exception as e:
#     print('Failed to set readout freq')
#     print(e)

exp_name = qubit.uid + "LocalSpecRF"

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=-500e6,
    stop=+500e6,
    count=201
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Drive Frequency Sweep',
        parameter=drive_freq_sweep,
    ):
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=400e-9, #qubit.parameters.user_defined["pulse_length"],
                    amplitude=0.1), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
            )
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.reserve('drive')
            exp.play(
                signal="measure",
                pulse=readout_pulse,)
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=2e-6,
                kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout",
                         play_after="single_res_point_readout",
                         length=1e-4): #qubit.parameters.user_defined['cw_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")
            exp.reserve(signal="drive")


exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency)-5e6,
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=qubit.parameters.drive_lo_frequency)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=100,)

## Plot and Analyze Data

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
# For plotting current vs single resonator point
freqs = my_acquired_results.axis[0] + qubit.parameters.drive_lo_frequency
# freqs = my_acquired_results.axis[0] + qubit.parameters.drive_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase - np.mean(phase)

fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Spectrum near Transition')
ax[0].set_xlabel('Drive Frequency (GHz)')
ax[0].set_ylabel('Amplitude')
ax[0].grid()
# ax[0].vlines(qubit.parameters.drive_frequency_ge+qubit.parameters.drive_lo_frequency, np.min(amplitude), np.max(amplitude), colors='r');
ax[1].plot(freqs, phase)
ax[1].set_title(f'{qubit.uid} Spectrum near Transition')
ax[1].set_xlabel('Drive Frequency (GHz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
# ax[0].vlines(qubit.parameters.resonance_frequency_ge, np.min(amplitude), np.max(amplitude), colors='r');
# ax[1].vlines(qubit.parameters.resonance_frequency_ge, np.min(phase), np.max(phase), colors='r')
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

# LF Qubit Spectroscopy (Not Working)
<1GHz drive frequency

In [ ]:
drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=370e6,
    stop=400e6,
    count=151
)

exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Drive Frequency Sweep',
        parameter=drive_freq_sweep,
    ):
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=qubit.parameters.user_defined["pulse_length"],
                    amplitude=1), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
            )
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.play(
                signal="measure",
                pulse=readout_pulse)
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=qubit.parameters.user_defined['readout_len'],
                kernel=readout_pulse #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="drive")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=(qubit.res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency)+0.2e6,
    modulation_type=ModulationType.SOFTWARE,)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_lo_lf,
    port_mode=PortMode.LF)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
# For plotting current vs single resonator point
drive_freqs = my_acquired_results.axis[0]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.angle(IQ_data)

fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(drive_freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Spectrum')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude')
ax[0].grid()
ax[1].plot(drive_freqs, phase)
ax[1].set_title(f'{qubit.uid} Spectrum')
ax[1].set_xlabel('Drive Frequency (GHz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
fig.tight_layout()

# X-Gate Amplitude Tuneup
Rabi Amplitude Sweep

In [ ]:
rabi_amp_sweep = LinearSweepParameter(
    uid='rabi_amplitude', start=0, stop=1, count=101)

exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Amplitude Sweep',
        parameter=rabi_amp_sweep,
    ):
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=100e-9, #qubit.parameters.user_defined["pulse_length"], #can have this be longer if not enough periods
                    amplitude=1,), #max power to start
                amplitude=rabi_amp_sweep,
            )
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.play(
                signal="measure",
                pulse=readout_pulse)
            exp.acquire(
                signal='acquire',
                handle='single_freq_data',
                kernel=readout_pulse #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout",
                         play_after='single_res_point_readout',
                         length=20e-6): #qubit.parameters.user_defined['time_domain_reset_length']*20):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")
            exp.reserve(signal="drive")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency) - 5e6,
    modulation_type=ModulationType.SOFTWARE,)
drive_osc = Oscillator(
    "drive_osc",
    frequency=4e7, #qubit.parameters.resonance_frequency_ge,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

# readout_osc = Oscillator(
#     "readout_osc",
#     frequency=6.8718e9-qubit.parameters.readout_lo_frequency, #(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency - 15e4)
#     modulation_type=ModulationType.SOFTWARE,)
# drive_osc = Oscillator(
#     "drive_osc",
#     frequency=qubit.parameters.drive_frequency_ge,
#     modulation_type=ModulationType.HARDWARE)
# drive_lo_lf = Oscillator(
#     "drive_lo_lf",
#     frequency=0)
# exp_calibration["drive"] = SignalCalibration(
#     oscillator=drive_osc,
#     local_oscillator=drive_lo_lf,
#     port_mode=PortMode.LF)
# exp_calibration["measure"] = SignalCalibration(
#     oscillator=readout_osc,)
# exp_calibration["acquire"] = SignalCalibration(
#     oscillator=readout_osc,
#     port_delay=qubit.parameters.readout_integration_delay)
# exp.set_calibration(exp_calibration)

## Compile Session
Do final checks before running the job.

In [ ]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=100,)

## Run session
- Think about how to implement session.queue() since this could allow the device to always be measuring something

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
# plot measurement data
drive_amp = my_acquired_results.axis[0]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase #-np.mean(phase)

fitting_plot_x = np.linspace(rabi_amp_sweep.start, rabi_amp_sweep.stop, 501)

try: popt_amp, pcov_amp = oscillatory.fit(drive_amp, amplitude)
except: pass

try: popt_phase, pcov_phase = oscillatory.fit(drive_amp, phase, 10, 0, 0.5, 0) #frequency, phase, amplitude, offset
except: pass

fig, ax = plt.subplots(2, 1, figsize=(8,6))
ax[0].plot(drive_amp, amplitude)
try: ax[0].plot(fitting_plot_x, oscillatory(fitting_plot_x, *popt_amp), '-r')
except: pass
ax[0].set_title(f'{qubit.uid} Amplitude Sweep')
ax[0].set_xlabel('Rabi Pulse Amplitude')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].vlines(0.81, ymin=np.min(amplitude), ymax=np.max(amplitude), color='orange')
ax[0].vlines(0.32, ymin=np.min(amplitude), ymax=np.max(amplitude), color='orange')
ax[0].grid()
ax[1].plot(drive_amp, phase)
try: ax[1].plot(fitting_plot_x, oscillatory(fitting_plot_x, *popt_phase), '-r')
except: pass
ax[1].set_title(f'{qubit.uid} Amplitude Sweep')
ax[1].set_xlabel('Rabi Pulse Amplitude')
ax[1].set_ylabel('Phase (a.u.)')
ax[1].vlines(qubit.parameters.user_defined['amplitude_pi']-0.01, ymin=np.min(phase), ymax=np.max(phase), color='orange')
ax[1].vlines(qubit.parameters.user_defined['amplitude_pi/2']-0.01, ymin=np.min(phase), ymax=np.max(phase), color='orange')
ax[1].grid()
fig.tight_layout()
try: print(f"Fitted parameters (amplitude): {popt_amp}")
except: pass
try: print(f"Fitted parameters (phase): {popt_phase}")
except: pass

# Amplitude Chevron 
(Amplitude and Pulse Length Sweeps)

In [ ]:
#--- Defining Pulse Shapes ---
readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.user_defined['readout_len'],
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)
drive_pulse_shape = pulse_library.gaussian_square(
    uid='Gaussian Square Pulse',
    length=1000e-9,
    amplitude=1)

#--- Defining Parameter Sweeps ---- 
drive_amp_sweep = SweepParameter(
    uid='Drive_Amp_Sweep',
    values=np.linspace(start=0, stop=1, num=101,))
drive_pulse_length_sweep = LinearSweepParameter(
    uid='Drive_Pulse_Time_Sweep',
    start=100e-9,
    stop=1100e-9,
    count=101)

#--- Signal Mappings ---
exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid='drive', map_to=lsg[qubit.uid]['drive_line'])]

#--- Defining Sections and Sweeps and Experiment ---
exp = Experiment(
    uid='Drive Chevron',
    signals=exp_signals,)

Drive_Amp_Sweep = Sweep(
    uid='Drive Amplitude Sweep',
    parameters=drive_amp_sweep)
Drive_Pulse_Length_Sweep = Sweep(
    uid='Drive Pulse Length Sweep',
    parameters=drive_pulse_length_sweep)

RT_Loop = AcquireLoopRt(
    uid='Shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,)

Drive = Section(uid='Driving Section')
Drive.play(
    signal='drive',
    pulse=drive_pulse_shape,
    length=drive_pulse_length_sweep,
    amplitude=drive_amp_sweep)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',
    play_after=Drive)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'])

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['time_domain_reset_length'],
    play_after=Meas_Acquire)

#--- Properly Defining Nesting Order ---
Outer_Sweep = Drive_Pulse_Length_Sweep
Inner_Sweep = Drive_Amp_Sweep
Outer_Sweep.execution_type = ExecutionType.NEAR_TIME
Inner_Sweep.execution_type = ExecutionType.REAL_TIME

exp.add(Outer_Sweep)
Outer_Sweep.add(RT_Loop)
RT_Loop.add(Inner_Sweep)
Inner_Sweep.add(Drive)
Inner_Sweep.add(Meas_Acquire)
Inner_Sweep.add(Delay_After_Count)

#--- Compile Session ---
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

#--- Run Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data']

## Plot and Analyze Data

In [ ]:
outer_sweep_param = my_acquired_results.axis[0]
inner_sweep_param = my_acquired_results.axis[1]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.angle(IQ_data)

fig, ax = plt.subplots(1,2, figsize=(15,6))
cmap0 = ax[0].pcolor(inner_sweep_param,
    outer_sweep_param,
    amplitude,
    shading='nearest')
ax[0].set_title(f'{qubit.uid} Resonator Current Response')
ax[0].set_xlabel(f'{Inner_Sweep.uid}')
ax[0].set_ylabel(f'{Outer_Sweep.uid}')
cmap1 = ax[1].pcolor(inner_sweep_param,
    outer_sweep_param,
    phase,
    shading='nearest',)
cbar0 = fig.colorbar(cmap0, ax=ax[0],)
cbar1 = fig.colorbar(cmap1, ax=ax[1])
cbar0.set_label('Measured Amplitude (a.u.)')
cbar1.set_label('Phase')
ax[1].set_title(f'{qubit.uid} Resonator Current Response')
ax[1].set_xlabel(f'{Inner_Sweep.uid}')
ax[1].set_ylabel(f'{Outer_Sweep.uid}')
fig.tight_layout()

# Detuning Chevron
(Pulse Length and Drive Frequency Sweeps)

## Experiment

In [ ]:
#--- Defining Pulse Shapes ---
readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.user_defined['readout_len'],
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)
drive_pulse_shape = pulse_library.gaussian_square(
    uid='Gaussian Square Pulse',
    length=1000e-9,
    amplitude=1)

#--- Defining Parameter Sweeps ---- 
drive_freq_sweep = LinearSweepParameter(
    uid='drive_freq_sweep',
    start=qubit.parameters.drive_frequency_ge-10e6,
    stop=qubit.parameters.drive_frequency_ge+10e6,
    count=101)
drive_pulse_length_sweep = LinearSweepParameter(
    uid='drive_pulse_time_sweep',
    start=100e-9,
    stop=1600e-9,
    count=101)

#--- Signal Mappings ---
exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid='drive', map_to=lsg[qubit.uid]['drive_line'])]

#--- Defining Sections and Sweeps and Experiment ---
exp = Experiment(
    uid='Drive Chevron',
    signals=exp_signals,)

Drive_Freq_Sweep = Sweep(
    uid='Drive Frequency Sweep',
    parameters=drive_freq_sweep)
Drive_Pulse_Length_Sweep = Sweep(
    uid='Drive Pulse Length Sweep',
    parameters=drive_pulse_length_sweep)

RT_Loop = AcquireLoopRt(
    uid='Shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,)

Drive = Section(uid='Driving Section')
Drive.play(
    signal='drive',
    pulse=drive_pulse_shape,
    length=drive_pulse_length_sweep,
    amplitude=1)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',
    play_after=Drive)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'])

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['time_domain_reset_length'],
    play_after=Meas_Acquire)

#--- Properly Defining Nesting Order ---
Outer_Sweep = Drive_Pulse_Length_Sweep
Inner_Sweep = Drive_Freq_Sweep
Outer_Sweep.execution_type = ExecutionType.NEAR_TIME
Inner_Sweep.execution_type = ExecutionType.REAL_TIME

exp.add(Outer_Sweep)
Outer_Sweep.add(RT_Loop)
RT_Loop.add(Inner_Sweep)
Inner_Sweep.add(Drive)
Inner_Sweep.add(Meas_Acquire)
Inner_Sweep.add(Delay_After_Count)

#--- Experiment Calibration ---
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration = Calibration()
exp_calibration["drive"] = SignalCalibration(oscillator=drive_osc,)
exp.set_calibration(exp_calibration)

#--- Compile Session ---
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

#--- Run Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data']

## Plot/Data Analysis

In [ ]:
outer_sweep_param = my_acquired_results.axis[0]
inner_sweep_param = my_acquired_results.axis[1]
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.angle(IQ_data)

fig, ax = plt.subplots(1,2, figsize=(15,6))
cmap0 = ax[0].pcolor(inner_sweep_param,
    outer_sweep_param,
    amplitude,
    shading='nearest')
ax[0].set_title(f'{qubit.uid} Resonator Current Response')
ax[0].set_xlabel(f'{Inner_Sweep.uid}')
ax[0].set_ylabel(f'{Outer_Sweep.uid}')
cmap1 = ax[1].pcolor(inner_sweep_param,
    outer_sweep_param,
    phase,
    shading='nearest',)
cbar0 = fig.colorbar(cmap0, ax=ax[0],)
cbar1 = fig.colorbar(cmap1, ax=ax[1])
cbar0.set_label('Measured Amplitude (a.u.)')
cbar1.set_label('Phase')
ax[1].set_title(f'{qubit.uid} Resonator Current Response')
ax[1].set_xlabel(f'{Inner_Sweep.uid}')
ax[1].set_ylabel(f'{Outer_Sweep.uid}')
fig.tight_layout()

# Qubit T1

In [ ]:
time_delay = LinearSweepParameter(
    uid='time_delay', start=0, stop=10e-6, count=15)

# def T1_1shot():
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Delay_sweep',
        parameter=time_delay,
    ):
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=qubit.parameters.user_defined["pulse_length"],
                    amplitude=qubit.parameters.user_defined['amplitude_pi']), #1,) #max power to start
            )
            exp.delay('drive', time_delay)
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.play(
                signal="measure",
                pulse=readout_pulse)
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=qubit.parameters.user_defined['readout_len'],
                kernel=readout_pulse #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="drive")

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency) - 5e6,
    modulation_type=ModulationType.HARDWARE,)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_lo_lf)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
    # return exp

# exp = T1_1shot()

## Compile Session
Do final checks before running the job.

In [ ]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

## Run session
- Think about how to implement session.queue() since this could allow the device to always be measuring something

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
amplitude = np.abs(my_acquired_results.data)
phase = np.unwrap(np.angle(my_acquired_results.data))
phase = phase - np.mean(phase)

delay_plot = np.linspace(time_delay.start, time_delay.stop, 501)
popt, pcov = exponential_decay.fit(time_delay, phase, 1/40e-6, 10, 7, plot=False)

fig, ax = plt.subplots(1,1, figsize=(4,4))
ax.plot(time_delay*1e6, phase, '.k')
ax.plot(delay_plot*1e6, exponential_decay(delay_plot, *popt), '-r');
ax.set_title(f"{qubit.uid}'s T1")
ax.set_xlabel('Delay (us)')
ax.set_ylabel('Amplitude')
ax.grid()

print(f"Fitted parameters: {popt}")
print('T1 time ' + str(1/popt[0]*1e6) + ' us') 

## T1 Statistics
Still need to change this to be of a matching format to the T2E and T2*

In [ ]:
def T1_run():
    exp = T1_1shot()
    session = Session(
        device_setup=device_setup,
        log_level = logging.WARNING,
        experiment=exp)
    session.connect();
    compiled_session = session.compile(exp);
    # psv.interactive_psv(compiled_session, max_events_to_publish=100,)
    
    results = session.run()
    my_results = session.get_results() #a deep copy of session.results
    my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

    IQ_data = my_acquired_results.data
    amplitude = np.abs(IQ_data)
    phase = np.unwrap(np.angle(IQ_data))
    phase = phase-np.mean(phase)
    
    try:
        popt_amp, pcov_amp = exponential_decay.fit(time_delay, amplitude, 1/40e-6, 10, 7, plot=False)
        print(f'Fitted Parameters (amplitude): {popt_amp}')
        print('T1 time ' + str(1/popt_amp[0]*1e6) + ' us')
        amp_t1 = popt_amp[0]
    except Exception as e:
        amp_t1 = None
    
    try:
        popt_phase, pcov_phase = exponential_decay.fit(time_delay, phase, 5e3, 0.2, -0.7, plot=False)
        print(f'Fitted Parameters (phase): {popt_phase}')
        print('T1 time ' + str(1/popt_phase[0]*1e6) + ' us')
        phase_t1 = popt_phase[0]
    except Exception as e: 
        phase_t1 = None
        
    return [amp_t1, phase_t1]

In [ ]:
datetime_array, T1_data = measure_data_stats(T1_run, n_runs=1000)

In [ ]:
T1_plotted = T1_data[1,:]
stat_out(1/T1_plotted*1e6, 'T1')
plot_box_whisker_time_series(datetime_array, 1/T1_plotted*1e6, 'T1', time_interval_str='10min')

# Qubit T2*
Ramsey Measurement

In [ ]:
time_delay = LinearSweepParameter(
    uid='time_delay', start=0, stop=60e-6, count=121)

ramsey_drive = pulse_library.gaussian_square(
    uid='ramsey_drive',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi/2'],
)

def T2star():
    exp = Experiment(
        uid="Spectroscopy",
        signals=[
            ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
            ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
            ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
        ]
    )
    
    with exp.acquire_loop_rt(
        uid='shots',
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        with exp.sweep(
            uid='Delay_sweep',
            parameter=time_delay,
        ):
            with exp.section(uid='Play drive'):
                exp.play(signal="drive", pulse=ramsey_drive)
                exp.delay('drive', time_delay)
                exp.play(signal="drive", pulse=ramsey_drive)
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=qubit.parameters.user_defined['readout_len'],
                    kernel=readout_pulse #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
                exp.reserve(signal="measure")
                exp.reserve(signal="drive")
    
    exp_calibration = Calibration()
    drive_osc = Oscillator(
        "drive_osc",
        frequency=qubit.parameters.drive_frequency_ge+0.15e6, #offset here of about 1MHz to 3MHz to get precise frequency
        modulation_type=ModulationType.HARDWARE)
    exp_calibration["drive"] = SignalCalibration(
        oscillator=drive_osc,)
    exp.set_calibration(exp_calibration)

    return exp

exp = T2star()

## Compile Session
Do final checks before running the job.

In [ ]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

## Run session
- Think about how to implement session.queue() since this could allow the device to always be measuring something

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase-np.mean(phase)

fitting_plot_x = np.linspace(time_delay.start, time_delay.stop, 501)

try: popt_amp, pcov_amp = oscillatory_decay.fit(time_delay, amplitude, 2e6, 0, 1e5, 0.1, 3)
except: pass

try: popt_phase, pcov_phase = oscillatory_decay.fit(time_delay, phase, 1e6, 0, 1e6, 0.5, 0) #frequency, phase, decay_rate, amplitude, offset
except: pass

fig, ax = plt.subplots(2, 1, figsize=(8,6))
ax[0].scatter(time_delay*1e6, amplitude)
try: ax[0].plot(fitting_plot_x*1e6, oscillatory_decay(fitting_plot_x, *popt_amp), '-r')
except: pass
ax[0].set_title(f'{qubit.uid} Ramsey Oscillations')
ax[0].set_xlabel('Time Delay (us)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].scatter(time_delay*1e6, phase)
try: ax[1].plot(fitting_plot_x*1e6, oscillatory_decay(fitting_plot_x, *popt_phase), '-r')
except: pass
ax[1].set_title(f'{qubit.uid} Ramsey Oscillations')
ax[1].set_xlabel('Time Delay (us)')
ax[1].set_ylabel('Phase (a.u.)')
ax[1].grid()
fig.tight_layout()
try:
    print(f"Fitted parameters (amplitude): {popt_amp}")
    print(f'detuning = {popt_amp[0]*1e-6} MHz, T2r = {1e6/popt_amp[2]} us')
except: pass
try:
    print(f"Fitted parameters (phase): {popt_phase}")
    print(f'detuning = {popt_phase[0]*1e-6} MHz, T2r = {1e6/popt_phase[2]} us')
except: pass

## Ramsey Statistics

In [ ]:
def T2star_run():
    exp = T2star()
    
    session = Session(
        device_setup=device_setup,
        log_level = logging.WARNING,
        experiment=exp)
    session.connect();
    compiled_session = session.compile(exp);
    
    results = session.run()
    my_results = session.get_results() #a deep copy of session.results
    my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name
    
    IQ_data = my_acquired_results.data
    amplitude = np.abs(IQ_data)
    phase = np.unwrap(np.angle(IQ_data))
    phase = phase-np.mean(phase)
    
    try:
        popt_amp, pcov_amp = oscillatory_decay.fit(time_delay, amplitude, 2e6, 0, 1e5, 0.1, 3)
        print(f'Fitted Parameters (amplitude): {popt_amp}')
        print('T2* time ' + str(1/popt_amp[2]*1e6) + ' us')
        amp_t2star = np.abs(popt_amp[2])
    except Exception as e:
        amp_t2star = None
    
    try:
        popt_phase, pcov_phase = oscillatory_decay.fit(time_delay, phase, 2.0e6, 1, 3e4, 0.2, 0, plot=False) #frequency, phase, decay_rate, amplitude, offset
        print(f'Fitted Parameters (phase): {popt_phase}')
        print('T2* time ' + str(1/popt_phase[2]*1e6) + ' us')
        phase_t2star = np.abs(popt_phase[2])
    except: 
        phase_t2star = None
        
    return [amp_t2star, phase_t2star]

In [ ]:
datetime_array, T2star_data = measure_data_stats(T2star_run, n_runs=1000)

In [ ]:
boolean_filt = (T2star_data[1,:] > 1e4)
T2star_data_filtered = np.copy(T2star_data)
T2star_data_filtered[1,:] = np.where(boolean_filt, T2star_data[1,:], None) 
print(1/T2star_data_filtered[1,:40]*1e6)

In [ ]:
T2star_plotted = T2star_data_filtered[1,:]
stat_out(1/T2star_plotted*1e6, 'T2*')
plot_box_whisker_time_series(datetime_array, 1/T2star_plotted*1e6, 'T2*', time_interval_str='20min')

# T2 Echo

In [ ]:
time_delay = LinearSweepParameter(
    uid='time_delay', start=0, stop=150e-6, count=16)

x90 = pulse_library.gaussian_square(
    uid='x90',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi/2']
)

y180 = pulse_library.gaussian_square(
    uid='y180',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi']
)

def T2Echo():
    exp = Experiment(
        uid="Spectroscopy",
        signals=[
            ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
            ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
            ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
        ]
    )
    
    with exp.acquire_loop_rt(
        uid='shots',
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        reset_oscillator_phase=True
    ):
        with exp.sweep(
            uid='Delay_sweep',
            parameter=time_delay,
            alignment=SectionAlignment.RIGHT,
            reset_oscillator_phase=True
        ):
            with exp.section(uid='Play drive', alignment=SectionAlignment.RIGHT):
                exp.play(signal="drive", phase=0, pulse=x90)
                exp.delay('drive', time_delay/2)
                exp.play(signal="drive", phase=np.pi, pulse=y180)
                exp.delay('drive', time_delay/2)
                exp.play(signal='drive', phase=0, pulse=x90)
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse)
                exp.acquire( 
                    signal='acquire',
                    handle='single_freq_data',
                    length=qubit.parameters.user_defined['readout_len'],
                    kernel=readout_pulse
                )
            with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
                exp.reserve(signal="measure")
                exp.reserve(signal="drive")
    
    exp_calibration = Calibration()
    drive_osc = Oscillator(
        "drive_osc",
        frequency=qubit.parameters.drive_frequency_ge,
        modulation_type=ModulationType.HARDWARE)
    exp_calibration["drive"] = SignalCalibration(
        oscillator=drive_osc,)
    exp.set_calibration(exp_calibration)

    return exp
    
exp = T2Echo()x90 = pulse_library.gaussian_square(
    uid='x90',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi/2']
)

y180 = pulse_library.gaussian_square(
    uid='y180',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi']
)

def T2Echo():
    exp = Experiment(
        uid="Spectroscopy",
        signals=[
            ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
            ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
            ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
        ]
    )
    
    with exp.acquire_loop_rt(
        uid='shots',
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        reset_oscillator_phase=True
    ):
        with exp.sweep(
            uid='Delay_sweep',
            parameter=time_delay,
            alignment=SectionAlignment.RIGHT,
            reset_oscillator_phase=True
        ):
            with exp.section(uid='Play drive', alignment=SectionAlignment.RIGHT):
                exp.play(signal="drive", phase=0, pulse=x90)
                exp.delay('drive', time_delay/2)
                exp.play(signal="drive", phase=np.pi, pulse=y180)
                exp.delay('drive', time_delay/2)
                exp.play(signal='drive', phase=0, pulse=x90)
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse)
                exp.acquire( 
                    signal='acquire',
                    handle='single_freq_data',
                    length=qubit.parameters.user_defined['readout_len'],
                    kernel=readout_pulse
                )
            with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
                exp.reserve(signal="measure")
                exp.reserve(signal="drive")
    
    exp_calibration = Calibration()
    drive_osc = Oscillator(
        "drive_osc",
        frequency=qubit.parameters.drive_frequency_ge,
        modulation_type=ModulationType.HARDWARE)
    exp_calibration["drive"] = SignalCalibration(
        oscillator=drive_osc,)
    exp.set_calibration(exp_calibration)

    return exp
    
exp = T2Echo()x90 = pulse_library.gaussian_square(
    uid='x90',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi/2']
)

y180 = pulse_library.gaussian_square(
    uid='y180',
    length=qubit.parameters.user_defined['pulse_length'],
    amplitude=qubit.parameters.user_defined['amplitude_pi']
)

def T2Echo():
    exp = Experiment(
        uid="Spectroscopy",
        signals=[
            ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
            ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
            ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])
        ]
    )
    
    with exp.acquire_loop_rt(
        uid='shots',
        count=averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        reset_oscillator_phase=True
    ):
        with exp.sweep(
            uid='Delay_sweep',
            parameter=time_delay,
            alignment=SectionAlignment.RIGHT,
            reset_oscillator_phase=True
        ):
            with exp.section(uid='Play drive', alignment=SectionAlignment.RIGHT):
                exp.play(signal="drive", phase=0, pulse=x90)
                exp.delay('drive', time_delay/2)
                exp.play(signal="drive", phase=np.pi, pulse=y180)
                exp.delay('drive', time_delay/2)
                exp.play(signal='drive', phase=0, pulse=x90)
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse)
                exp.acquire( 
                    signal='acquire',
                    handle='single_freq_data',
                    length=qubit.parameters.user_defined['readout_len'],
                    kernel=readout_pulse
                )
            with exp.section(uid="delay_between_readout", length=qubit.parameters.user_defined['time_domain_reset_length']):
                exp.reserve(signal="measure")
                exp.reserve(signal="drive")
    
    exp_calibration = Calibration()
    drive_osc = Oscillator(
        "drive_osc",
        frequency=qubit.parameters.drive_frequency_ge,
        modulation_type=ModulationType.HARDWARE)
    exp_calibration["drive"] = SignalCalibration(
        oscillator=drive_osc,)
    exp.set_calibration(exp_calibration)

    return exp
    
exp = T2Echo()

## Compile Session
Do final checks before running the job.

In [ ]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

## Run session
- Think about how to implement session.queue() since this could allow the device to always be measuring something

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

## Plot and Analyze Data

In [ ]:
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase-np.mean(phase)

fitting_plot_x = np.linspace(time_delay.start, time_delay.stop, 501)

try: popt_amp, pcov_amp = exponential_decay.fit(time_delay, amplitude, 1e6, 0 , 1)
except: pass

try: popt_phase, pcov_phase = exponential_decay.fit(time_delay, phase, 1e6, 0, 1) #decay rate, offset, amplitude
except: pass

fig, ax = plt.subplots(2, 1, figsize=(4,8))
ax[0].plot(time_delay*1e6, amplitude, '.k')
try: ax[0].plot(fitting_plot_x*1e6, exponential_decay(fitting_plot_x, *popt_amp), '-r')
except: pass
ax[0].set_title(f'{qubit.uid} T2 Echo')
ax[0].set_xlabel('Time Delay (us)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].plot(time_delay*1e6, phase, '.k')
try: ax[1].plot(fitting_plot_x*1e6, exponential_decay(fitting_plot_x, *popt_phase), '-r')
except: pass
ax[1].set_title(f'{qubit.uid} T2 Echo')
ax[1].set_xlabel('Time Delay (us)')
ax[1].set_ylabel('Phase (a.u.)')
ax[1].grid()
fig.tight_layout()
try:
    print(f"Fitted parameters (amplitude): {popt_amp}")
    print('T2e time ' + str(1/popt_amp[0]*1e6) + ' us') 
except: pass
try:
    print(f"Fitted parameters (phase): {popt_phase}")
    print('T2e time ' + str(1/popt_phase[0]*1e6) + ' us') 
except: pass

In [ ]:
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = np.unwrap(np.angle(IQ_data))
phase = phase-np.mean(phase)

fitting_plot_x = np.linspace(time_delay.start, time_delay.stop, 501)

try: popt_amp, pcov_amp = exponential_decay.fit(time_delay, amplitude, 1e6, 0 , 1)
except: pass

try: popt_phase, pcov_phase = exponential_decay.fit(time_delay, phase, 1e6, 0, 1) #decay rate, offset, amplitude
except: pass

fig, ax = plt.subplots(2, 1, figsize=(4,8))
ax[0].plot(time_delay, amplitude, '.k')
try: ax[0].plot(fitting_plot_x, exponential_decay(fitting_plot_x, *popt_amp), '-r')
except: pass
ax[0].set_title(f'{qubit.uid} T2 Echo')
ax[0].set_xlabel('Time Delay (s)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].plot(time_delay, phase, '.k')
try: ax[1].plot(fitting_plot_x, exponential_decay(fitting_plot_x, *popt_phase), '-r')
except: pass
ax[1].set_title(f'{qubit.uid} T2 Echo')
ax[1].set_xlabel('Time Delay (s)')
ax[1].set_ylabel('Phase (a.u.)')
ax[1].grid()
fig.tight_layout()
try:
    print(f"Fitted parameters (amplitude): {popt_amp}")
    print('T2e time ' + str(1/popt_amp[0]*1e6) + ' us') 
except: pass
try:
    print(f"Fitted parameters (phase): {popt_phase}")
    print('T2e time ' + str(1/popt_phase[0]*1e6) + ' us') 
except: pass

## T2E Statistics

In [ ]:
def T2E_run():
    exp = T2Echo()
    session = Session(
        device_setup=device_setup,
        log_level = logging.WARNING,
        experiment=exp)
    session.connect();
    compiled_session = session.compile(exp);
    
    results = session.run()
    my_results = session.get_results() #a deep copy of session.results
    my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name
    
    IQ_data = my_acquired_results.data
    amplitude = np.abs(IQ_data)
    phase = np.unwrap(np.angle(IQ_data))
    phase = phase-np.mean(phase)
    
    fitting_plot_x = np.linspace(time_delay.start, time_delay.stop, 501)
    
    try:
        popt_amp, pcov_amp = exponential_decay.fit(time_delay, amplitude, 1e6, 0 , 1)
        print(f"Fitted Parameters (amplitude): {popt_amp}")
        print('T2e time ' + str(1/popt_amp[0]*1e6) + ' us')
        amp_t2e = popt_amp[0]
    except Exception as e:
        amp_t2e = None
    try:
        popt_phase, pcov_phase = exponential_decay.fit(time_delay, phase, 1e6, 0, 1) #decay rate, offset, amplitude
        print(f"Fitted Parameters (phase): {popt_phase}")
        print('T2e time ' + str(1/popt_phase[0]*1e6) + ' us')
        phase_t2e = popt_phase[0]
    except Exception as e: 
        phase_t2e = None
        
    return [amp_t2e, phase_t2e]

In [ ]:
datetime_array, T2E_data = measure_data_stats(T2E_run, n_runs=300)

In [ ]:
T2E_plotted = T2E_data[1,:]
stat_out(1/T2E_plotted*1e6, 'T2E')
plot_box_whisker_time_series(datetime_array, 1/T2E_plotted*1e6, 'T2E', time_interval_str='5min')

# Qubit 1D Flux Spectrum

In [ ]:
exp_name = qubit.uid + "1DFluxSpecRF"

current_sweep_vals = np.linspace(-120e-6, -105e-6, 31)

current_sweep = SweepParameter(
    uid='Current_Sweep',
    values=current_sweep_vals)

try:
    ro_values = (res_to_current(current_sweep_vals)-
        qubit.parameters.readout_lo_frequency)
    print('Using resonator frequency map')
except:
    ro_values = np.full(
        current_sweep_vals.shape,
        qubit.parameters.readout_frequency)
    print('Using constant resonator frequency')

readout_freq = SweepParameter(
    uid='RO_Pulse',
    values=ro_values)
    
drive_lo_freq_sweep = LinearSweepParameter(
    uid='Drive_LO',
    start=4e9,
    stop=4e9,
    count=1)

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=-500e6,
    stop=498e6,
    count=501)

drive_pulse = pulse_library.gaussian(
    uid="drive_pulse",
    length=40e-9,
    sigma=0.25)
'''
drive_pulse = pulse_library.gaussian_square(
    uid="drive_pulse",
    length=50e-9, #qubit.parameters.user_defined['pulse_length'],
    width=40e-9, #qubit.parameters.user_defined['pulse_length']*0.9,
    sigma=0.2,)
'''

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])]

exp = Experiment(
    uid="Spectroscopy",
    signals=exp_signals)

Current_RO_Sweep = Sweep(
    uid=current_sweep.uid,
    parameters=[readout_freq, current_sweep],
    execution_type=ExecutionType.NEAR_TIME)
Current_RO_Sweep.call(
    change_coil_current,
    new_current=current_sweep,
    step_time=0.01)

Drive_LO_Freq_Sweep = Sweep(
    uid=drive_lo_freq_sweep.uid,
    parameters=drive_lo_freq_sweep,
    execution_type=ExecutionType.NEAR_TIME)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION)

Drive_Freq_Sweep = Sweep(
    uid=drive_freq_sweep.uid,
    parameters=drive_freq_sweep,
    execution_type=ExecutionType.REAL_TIME)

Drive = Section(uid = 'Drive')
Drive.play(signal='drive', pulse=drive_pulse, amplitude=1)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',
    play_after=Drive)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'],
    kernel=readout_pulse)

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=100e-6, #qubit.parameters.user_defined['time_domain_reset_length'],
    play_after=Meas_Acquire)

exp.add(Current_RO_Sweep)
Current_RO_Sweep.add(Drive_LO_Freq_Sweep)
Drive_LO_Freq_Sweep.add(RT_Loop)
RT_Loop.add(Drive_Freq_Sweep)
Drive_Freq_Sweep.add(Drive)
Drive_Freq_Sweep.add(Meas_Acquire)
Drive_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()

readout_osc = Oscillator(
    "readout_osc",
    frequency=readout_freq,
    modulation_type=ModulationType.SOFTWARE,)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)

drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
drive_lo = Oscillator(
    'drive_lo',
    frequency=drive_lo_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_lo)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
session.register_neartime_callback(change_coil_current)
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=100,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
drive_AWG_freqs = my_acquired_results.axis[2]
drive_lo_array = my_acquired_results.axis[1]
drive_freqs = np.empty((0))
for drive_lo in drive_lo_array:
    drive_freqs = np.append(drive_freqs, drive_lo + drive_AWG_freqs,)

currents = my_acquired_results.axis[0][1]

data = my_acquired_results.data
freq_shape = my_acquired_results.data.shape[1]*my_acquired_results.data.shape[2]
shape_tuple = (currents.shape[0], freq_shape)
IQ_data = data.reshape(shape_tuple)
IQ_data = IQ_data - np.mean(IQ_data, axis=1)[:,None]
amplitude = np.abs(IQ_data)
# phase = np.angle(IQ_data)
phase = adjust_phase(IQ_data, drive_freqs, qubit.parameters.readout_integration_delay)
phase = phase - np.mean(phase, axis=1)[:, None]
fig, ax = plt.subplots(1,2, figsize=(8,5))
cmap0 = ax[0].pcolor(currents*1e6,
    drive_freqs,
    amplitude.T,
    shading='nearest',)
ax[0].set_title(f'{qubit.uid} Two Tone Spectroscopy')
ax[0].set_xlabel('Currents (uA)')
ax[0].set_ylabel('Drive Frequency (GHz)')
cmap1 = ax[1].pcolor(currents*1e6,
    drive_freqs,
    phase.T,
    shading='nearest',)
fig.colorbar(cmap0, ax=ax[0])
ax[1].set_title(f'{qubit.uid} Two Tone Spectroscopy')
ax[1].set_xlabel('Currents (uA)')
ax[1].set_ylabel('Drive Frequency (GHz)')
fig.colorbar(cmap1, ax=ax[1])
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
#Define the Gaussian function 
def gauss(x, H, A, x0, sigma): 
    return H + A * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))
popt_array = []
pcov_array = []
p0 = [0.05, 0.2, 3.6e9, 1.5e7]
for i, _ in enumerate(currents):
    (popt, pcov) = curve_fit(gauss, drive_freqs, amplitude[i], p0=p0);
    p0 = popt
    # p0[2] = p0[2] + 0.05e9
    pcov_array.append(np.diag(pcov)[2])
    popt_array.append(popt[2])
plt.scatter(currents*1e6, popt_array)

# Qubit 1D Flux Spectrum LF

## Experiment

In [ ]:
exp_name = qubit.uid + "1DFluxSpecLF"

# current_sweep_vals = np.linspace(-10e-6, 10e-6, 5)
current_sweep_vals = np.linspace(-178e-6, -162e-6, 17)

current_sweep = SweepParameter(
    uid='Current_Sweep',
    values=current_sweep_vals)

try:
    ro_values = (
        res_to_current(current_sweep_vals)-
        qubit.parameters.readout_lo_frequency)
    print('Using resonator frequency map')
except:
    ro_values = np.full(
        current_sweep_vals.shape,
        qubit.parameters.readout_frequency)
    print('Using constant resonator frequency')

readout_freq = SweepParameter(
    uid='RO_Pulse',
    values=ro_values)
    
drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=0e6,
    stop=1000e6,
    count=101)

drive_pulse = pulse_library.gaussian_square(
    uid="drive_pulse",
    length=qubit.parameters.user_defined['pulse_length'],
    width=qubit.parameters.user_defined['pulse_length']*0.9,
    sigma=0.2,)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line'])]

exp = Experiment(
    uid="Spectroscopy",
    signals=exp_signals)

Current_RO_Sweep = Sweep(
    uid=current_sweep.uid,
    parameters=[readout_freq, current_sweep],
    execution_type=ExecutionType.NEAR_TIME)
Current_RO_Sweep.call(
    change_coil_current,
    new_current=current_sweep,
    step_time=0.01)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION)

Drive_Freq_Sweep = Sweep(
    uid=drive_freq_sweep.uid,
    parameters=drive_freq_sweep,
    execution_type=ExecutionType.REAL_TIME)

Drive = Section(uid = 'Drive')
Drive.play(signal='drive', pulse=drive_pulse, amplitude=1)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout',
    play_after=Drive)
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'],
    kernel=readout_pulse)

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=10e-6, #qubit.parameters.user_defined['time_domain_reset_length'],
    play_after=Meas_Acquire)

exp.add(Current_RO_Sweep)
Current_RO_Sweep.add(RT_Loop)
RT_Loop.add(Drive_Freq_Sweep)
Drive_Freq_Sweep.add(Drive)
Drive_Freq_Sweep.add(Meas_Acquire)
Drive_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()

readout_osc = Oscillator(
    "readout_osc",
    frequency=readout_freq,
    modulation_type=ModulationType.SOFTWARE,)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)

drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
drive_LF_osc = Oscillator(
    "drive_LF_osc",
    frequency=0,)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    local_oscillator=drive_LF_osc,
    port_mode=PortMode.LF)

exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
session.register_neartime_callback(change_coil_current)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
drive_freqs = my_acquired_results.axis[1]
currents = my_acquired_results.axis[0][1]
data = my_acquired_results.data
freqs = my_acquired_results.axis[0][0]+qubit.parameters.readout_lo_frequency
# freqs = np.tile(freqs, (201, 1)).T
# print(freqs[0,:])

shape_tuple = (currents.shape[0], drive_freqs.shape[0])
IQ_data = data.reshape(shape_tuple)
IQ_data = IQ_data - np.mean(IQ_data, axis=1)[:,None]
print(IQ_data.shape)
amplitude = np.abs(IQ_data)
phase = np.angle(IQ_data)
# phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)
phase = phase - np.mean(phase, axis=1)[:, None]
fig, ax = plt.subplots(1,2, figsize=(16,9))
cmap0 = ax[0].pcolor(currents*1e6,
    drive_freqs,
    amplitude.T,
    shading='nearest',)
    # vmax=3)
    # vmax=0.5)
    # vmin=0,
    # vmax=1)
ax[0].set_title(f'{qubit.uid} Two Tone Spectroscopy')
ax[0].set_xlabel('Currents (uA)')
ax[0].set_ylabel('Drive Frequency (GHz)')
cmap1 = ax[1].pcolor(currents*1e6,
    drive_freqs,
    phase.T,
    shading='nearest',)
    # vmin=,
    # vmax=
# ax[0].axvline(x=-25, color='red', linestyle='--', linewidth=2)
fig.colorbar(cmap0, ax=ax[0])
ax[1].set_title(f'{qubit.uid} Two Tone Spectroscopy')
ax[1].set_xlabel('Currents (uA)')
ax[1].set_ylabel('Drive Frequency (GHz)')
fig.colorbar(cmap1, ax=ax[1])
fig.tight_layout()

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
IQ_data = data.reshape(shape_tuple)
IQ_data = IQ_data - np.mean(IQ_data, axis=1)[:,None]
amplitude = np.abs(IQ_data)
amplitude = amplitude - np.tile(np.expand_dims(np.mean(amplitude, axis=0), axis=0), reps=(amplitude.shape[0], 1))
amplitude = amplitude - np.tile(np.expand_dims(np.mean(amplitude, axis=1), axis=-1), reps=(1, amplitude.shape[1]))
phase = np.angle(IQ_data)
# phase = adjust_phase(IQ_data, drive_freqs, qubit.parameters.readout_integration_delay)
phase = phase - np.mean(phase, axis=1)[:, None]
fig, ax = plt.subplots(1,2, figsize=(35,25))
cmap0 = ax[0].pcolor(currents*1e6,
    drive_freqs,
    amplitude.T,
    shading='nearest',
    vmin=-0.1,
    vmax=0.6)
ax[0].set_title(f'{qubit.uid} Resonator Current Response')
ax[0].set_xlabel('Currents (uA)')
ax[0].set_ylabel('Drive Frequency (GHz)')
cmap1 = ax[1].pcolor(currents*1e6,
    drive_freqs,
    phase.T,
    shading='nearest',)
    # vmin=,
    # vmax=
ax[0].axvline(x=-25, color='red', linestyle='--', linewidth=2)
fig.colorbar(cmap0, ax=ax[0])
ax[1].set_title(f'{qubit.uid} Resonator Current Response')
ax[1].set_xlabel('Currents (uA)')
ax[1].set_ylabel('Drive Frequency (GHz)')
fig.colorbar(cmap1, ax=ax[1])
fig.tight_layout()

# Fast Flux Pulse

## Experimental Options

### Const duty cycle, drive post pulse

In [ ]:
exp_name = qubit.uid + "FFluxConstDutyPostPulse"

flux_pulse_time = 200e-6
sampling_rate = 2e9
#first column is I data, second is Q
#The Q appears as neg of entered value. If phase irrelevant, just do I
fast_flux_array = np.ones((int(flux_pulse_time*sampling_rate),2))*1 
fast_flux_pulse = pulse_library.sampled_pulse(
    samples=fast_flux_array,
    uid='Fast_Flux_Pulse',
    can_compress=True)

t_delay = SweepParameter(
    uid='t_delay',
    values=np.logspace(start=-9,stop=-2.6989, num=121,base=10))

t_total = 2e-3

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=0,
    stop=400e6,
    count=201
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='t_delay_sweep',
        execution_type=ExecutionType.REAL_TIME,
        parameter=t_delay,
    ):
        with exp.sweep(
            uid='Drive Frequency Sweep',
            execution_type=ExecutionType.REAL_TIME,
            parameter=drive_freq_sweep,
        ):
            with exp.section(uid='Play flux'):
                exp.play(signal='flux', pulse=fast_flux_pulse)
            with exp.section(uid='Play drive', play_after='Play flux'):
                exp.delay(signal='drive', time=t_delay)
                exp.play(
                    signal="drive",
                    pulse=pulse_library.gaussian_square(
                        uid=f"drive_spec_pulse_{qubit.uid}",
                        length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                        amplitude=0.18), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
                )
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse,)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=2e-6,
                    kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout",
                             play_after="Play flux",):
                exp.delay(signal="measure", time=t_total-t_delay)
                exp.reserve(signal="acquire")
                exp.reserve(signal="drive")


exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    range=5,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

### Null pulse

In [ ]:
exp_name = qubit.uid + "FFluxNullPulse"

flux_pulse_time = 100e-6
sampling_rate = 2e9
#first column is I data, second is Q
#The Q appears as neg of entered value. If phase irrelevant, just do I
fast_flux_array = np.ones((int(flux_pulse_time*sampling_rate),2))*0
fast_flux_pulse = pulse_library.sampled_pulse(
    samples=fast_flux_array,
    uid='Fast_Flux_Pulse',
    can_compress=True)

t_delay = SweepParameter(
    uid='t_delay',
    values=np.logspace(start=-9,stop=-3, num=7,base=10))

t_total = 150e-6

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=4e9-qubit.parameters.drive_lo_frequency-100e6,
    stop=4e9-qubit.parameters.drive_lo_frequency+100e6,
    count=51
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        # ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        # ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        # ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='t_delay_sweep',
        execution_type=ExecutionType.REAL_TIME,
        parameter=t_delay,
    ):
        with exp.sweep(
            uid='Drive Frequency Sweep',
            execution_type=ExecutionType.REAL_TIME,
            parameter=drive_freq_sweep,
        ):
            with exp.section(uid='Play flux'):
                exp.play(signal='flux', pulse=fast_flux_pulse)
            with exp.section(uid='Play drive', play_after='Play flux'):
                exp.delay(signal='drive', time=t_delay)
                exp.play(
                    signal="drive",
                    pulse=pulse_library.gaussian_square(
                        uid=f"drive_spec_pulse_{qubit.uid}",
                        length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                        amplitude=0.2), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
                )
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse,)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=2e-6,
                    kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout",
                             play_after="Play flux",):
                exp.delay(signal="measure", time=t_total-t_delay)
                exp.reserve(signal="acquire")
                exp.reserve(signal="drive")

exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    range=5,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

### Sweep through and after pulse, fixed duty cycle
Should see flipped response due to flux pulse

In [ ]:
exp_name = qubit.uid + "FFluxPulseConstDutySweepThru"

flux_pulse_time = 500e-6
sampling_rate = 2e9
#first column is I data, second is Q
#The Q appears as neg of entered value. If phase irrelevant, just do I
fast_flux_array = np.ones((int(flux_pulse_time*sampling_rate),2))*1
fast_flux_pulse = pulse_library.sampled_pulse(
    samples=fast_flux_array,
    uid='Fast_Flux_Pulse',
    can_compress=True)

t_delay = SweepParameter(
    uid='t_delay',
    values=np.linspace(start=0,stop=1e-3, num=51))

t_total = 1e-3

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=0,
    stop=400e6,
    count=51
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        # ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
        # ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['fast_flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='t_delay_sweep',
        execution_type=ExecutionType.REAL_TIME,
        parameter=t_delay,
    ):
        with exp.sweep(
            uid='Drive Frequency Sweep',
            execution_type=ExecutionType.REAL_TIME,
            parameter=drive_freq_sweep,
        ):
            with exp.section(uid='Play flux'):
                exp.play(signal='flux', pulse=fast_flux_pulse)
            with exp.section(uid='Play drive'):
                exp.delay(signal="drive", time=t_delay)
                exp.play(
                    signal="drive",
                    pulse=pulse_library.gaussian_square(
                        uid=f"drive_spec_pulse_{qubit.uid}",
                        length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                        amplitude=0.2), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
                )
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse,)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=2e-6,
                    kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout"):
                exp.delay(signal="measure", time=t_total-t_delay)
                exp.reserve(signal="acquire")
                exp.reserve(signal="drive")


exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    range=5,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

### Fixed pulse length, increased pulse off time
Effectively changes the duty cycle
T_delay is 0 (const)

In [ ]:
exp_name = qubit.uid + "FFluxPulseOffSweep"

flux_pulse_time = 10e-6
sampling_rate = 2e9
#first column is I data, second is Q
#The Q appears as neg of entered value. If phase irrelevant, just do I
fast_flux_array = np.ones((int(flux_pulse_time*sampling_rate),2))*1
fast_flux_pulse = pulse_library.sampled_pulse(
    samples=fast_flux_array,
    uid='Fast_Flux_Pulse',
    can_compress=True)

t_total = SweepParameter(
    uid='t_delay',
    values=np.logspace(start=-9,stop=-3, num=10,base=10))

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=4e9-qubit.parameters.drive_lo_frequency,
    stop=4e9-qubit.parameters.drive_lo_frequency+300e6,
    count=51
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='t_delay_sweep',
        execution_type=ExecutionType.REAL_TIME,
        parameter=t_total,
    ):
        with exp.sweep(
            uid='Drive Frequency Sweep',
            execution_type=ExecutionType.REAL_TIME,
            parameter=drive_freq_sweep,
        ):
            with exp.section(uid='Play flux'):
                exp.play(signal='flux', pulse=fast_flux_pulse)
            with exp.section(uid='Play drive', play_after='Play flux'):
                exp.play(
                    signal="drive",
                    pulse=pulse_library.gaussian_square(
                        uid=f"drive_spec_pulse_{qubit.uid}",
                        length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                        amplitude=0.2), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
                )
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse,)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=2e-6,
                    kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout",
                             play_after="Play flux",):
                exp.delay(signal="measure", time=t_total)
                exp.reserve(signal="acquire")
                exp.reserve(signal="drive")


exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    range=5,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

## Compile Session
Do final checks before running the job.

In [ ]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

## Run session
- Think about how to implement session.queue() since this could allow the device to always be measuring something

In [ ]:
session.run();

In [ ]:
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
t_delay_len = my_acquired_results.axis[0]
drive_rf = my_acquired_results.axis[1]
drive_rf = drive_rf+3.6e9 #should be session.__something__....drive_lo_freq
IQ_data = my_acquired_results.data
IQ_data = IQ_data - np.mean(IQ_data, axis=1)[:,None]
amplitude = np.abs(IQ_data)
fig, ax = plt.subplots(1,1, figsize=(8,5))
cmap0 = ax.pcolor(t_delay_len,
    drive_rf,
    amplitude.T,
    shading='nearest',)
ax.set_title(f'{qubit.uid} Fast Flux Pulse')
ax.set_xlabel('Time since start of 500us flux pulse(s)')
ax.set_ylabel('Drive Frequency (GHz)')
# ax.set_xscale('log')
fig.colorbar(cmap0, ax=ax)
fig.tight_layout()
fig.savefig(str(data_directory_update()) + '/F1_Fast_Flux_Pulse_log', dpi=800)

In [ ]:
session.save(non_redund_name(exp_name))

In [ ]:
freq_array = []
p0 = [0, 0.4, 3.85e9, 20e6]
# popt = np.load(str(data_directory_update()) + '\F1_Flux_Pulse_Freq_Curve.npy')
# plt.plot(drive_rf, popt)
for amp in amplitude:
    # (popt, pcov) = curve_fit(gauss, drive_rf, amp, p0=p0, method='lm')
    # freq_array.append(popt[2])
    # p0 = popt
    transition = drive_rf[np.where(amp==max(amp))]
    freq_array.append(transition[0])
    # freq_array.append(drive_rf[np.where(max(amp))])
plt.plot(t_delay, freq_array)
plt.ylabel('Transition Frequency');
plt.xlabel('T Delay (s)');
# plt.xscale('log')
# np.save(str(data_directory_update()) + '/F1_Flux_Pulse_Freq_Curve', popt)

In [ ]:
plt.plot(t_delay, flux_array)
plt.ylabel('Flux (uA)');
plt.xlabel('T Delay (s)');

In [ ]:
flux_array = []
for rf in freq_array:
    flux_array.append(approximate_inverse(currents[:16], popt_array[:16], rf)*1e6)

In [ ]:
(popt, pcov) = curve_fit(exp_decay, t_delay.values, flux_array, p0=[1.34e8, 7.02e3, 4.1e9])
print(popt)
print('Time constant:', 1/popt[1], 's')

## Heralding (Two Acquires)

### Temp:

In [ ]:
exp_name = qubit.uid + "FirstMultiAcquirePass"

# flux_pulse_time = 200e-6
sampling_rate = 2e9
#first column is I data, second is Q
#The Q appears as neg of entered value. If phase irrelevant, just do I
# fast_flux_array = np.ones((int(flux_pulse_time*sampling_rate),2))*1 
# fast_flux_pulse = pulse_library.sampled_pulse(
    # samples=fast_flux_array,
    # uid='Fast_Flux_Pulse',
    # can_compress=True)

# t_delay = SweepParameter(
    # uid='t_delay',
    # values=np.logspace(start=-9,stop=-2.6989, num=121,base=10))

# t_total = 2e-3

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=0,
    stop=400e6,
    count=201
)

ro_length = 2e-6
readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=ro_length, 
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=ro_length, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='t_delay_sweep',
        execution_type=ExecutionType.REAL_TIME,
        parameter=t_delay,
    ):
        with exp.sweep(
            uid='Drive Frequency Sweep',
            execution_type=ExecutionType.REAL_TIME,
            parameter=drive_freq_sweep,
        ):
            with exp.section(uid='Play flux'):
                exp.play(signal='flux', pulse=fast_flux_pulse)
            with exp.section(uid='Play drive', play_after='Play flux'):
                exp.delay(signal='drive', time=t_delay)
                exp.play(
                    signal="drive",
                    pulse=pulse_library.gaussian_square(
                        uid=f"drive_spec_pulse_{qubit.uid}",
                        length=50e-9, #qubit.parameters.user_defined["pulse_length"],
                        amplitude=0.18), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
                )
            with exp.section(
                uid="single_res_point_readout",
                play_after='Play drive',
            ):
                exp.play(
                    signal="measure",
                    pulse=readout_pulse,)
                exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                    signal='acquire',
                    handle='single_freq_data',
                    length=2e-6,
                    kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
                )
            with exp.section(uid="delay_between_readout",
                             play_after="Play flux",):
                exp.delay(signal="measure", time=t_total-t_delay)
                exp.reserve(signal="acquire")
                exp.reserve(signal="drive")


exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=3.6e9)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    range=5,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

In [ ]:
ro_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_frequency-10e6,
    stop=qubit.parameters.readout_frequency+10e6,
    count=201)

dc_sweep_param = LinearSweepParameter(
    uid='DC Current',
    start=(175.7-(212.7/2))*1e-6,
    stop=(175.7+(212.7/2))*1e-6,
    count=213)

coil_sweep_param = LinearSweepParameter(
    start=-300*1e-6,
    stop=300*1e-6,
    count=301)

current_sweep_param = coil_sweep_param

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line'])]

exp = Experiment(
    uid="1D Current vs Resonator",
    signals=exp_signals,)

Current_Sweep = Sweep(
    uid=current_sweep_param.uid,
    parameters=current_sweep_param,
    execution_type=ExecutionType.NEAR_TIME)
Current_Sweep.call(
    change_coil_current,
    new_current=current_sweep_param,
    step_time=0.01)

RT_Loop = AcquireLoopRt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)

RO_Freq_Sweep = Sweep(
    uid=ro_freq_sweep.uid,
    parameters=ro_freq_sweep,
    execution_type=ExecutionType.REAL_TIME,)

Meas_Acquire = Section(
    uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse,)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='RO_data', 
    length=qubit.parameters.user_defined['readout_len'])

Delay_After_Count = Section(
    uid='Delay Between Readout',
    length=qubit.parameters.user_defined['cw_reset_length'],
    play_after=Meas_Acquire)

exp.add(Current_Sweep)
Current_Sweep.add(RT_Loop)
RT_Loop.add(RO_Freq_Sweep)
RO_Freq_Sweep.add(Meas_Acquire)
RO_Freq_Sweep.add(Delay_After_Count)

exp_calibration = Calibration()

readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
session.register_neartime_callback(change_coil_current)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=10000,)

results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['RO_data'] #extracts data from the exp.acquire method with the same key name

In [ ]:
# qubit drive pulse
x90 = pulse_library.drag(uid="drag_pulse", length=400e-9, amplitude=1.0, beta=0.3)

# measure pulse
readout_pulse = pulse_library.const(uid="readout_pulse", length=200e-9, amplitude=1.0)
# readout integration weights
readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=200e-9, amplitude=0.5
)

In [ ]:
start = 0.1
stop = 1
count = 5
amplitude_sweep = LinearSweepParameter(
    uid="amplitude", start=start, stop=stop, count=count
)

In [ ]:
# Create Experiment
exp = Experiment(
    uid="Amplitude Rabi",
    signals=[
        ExperimentSignal("drive"),
        ExperimentSignal("measure"),
        ExperimentSignal("acquire"),
    ],
)
## experimental pulse sequence
# outer loop - real-time, cyclic averaging in standard integration mode
with exp.acquire_loop_rt(
    uid="shots",
    count=2**5,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    # inner loop - real-time sweep of qubit drive pulse amplitude
    with exp.sweep(
        uid="sweep", parameter=amplitude_sweep, alignment=SectionAlignment.RIGHT
    ):
        # qubit excitation - pulse amplitude will be swept
        with exp.section(uid="qubit_excitation", alignment=SectionAlignment.RIGHT):
            exp.play(signal="drive", pulse=x90, amplitude=amplitude_sweep)
        # qubit readout pulse and data acquisition
        with exp.section(uid="qubit_readout"):
            exp.reserve(signal="drive")
            # play readout pulse
            exp.play(signal="measure", pulse=readout_pulse)
            # signal data acquisition
            exp.acquire(
                signal="acquire",
                handle="ac_0",
                kernel=readout_weighting_function,
            )
        # relax time after readout - for signal processing and qubit relaxation to ground state
        with exp.section(uid="relax", length=1e-6):
            pass
            # exp.reserve(signal="measure")

In [ ]:
psv.interactive_psv(compiled_session, max_events_to_publish=100)

### Plain Jaine Flux Pulse From Before

In [ ]:
readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)


# def fast_flux_pulse_exp():
#     exp = Experiment(
#         uid="Spectroscopy",
#         signals=[
#             ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
#             ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
#             ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
#             ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
#         ]
#     )
#     with exp.acquire_loop_rt(
#         uid='shots',
#         count=averages,
#         averaging_mode=AveragingMode.CYCLIC,
#         acquisition_type=AcquisitionType.INTEGRATION,
#         reset_oscillator_phase=False
#     ):

#         with exp.sweep(uid='Drive Frequency Sweep', parameter=drive_freq_sweep):
#             with exp.section(uid='Drive and Readout'):
#                 with exp.section(uid='Flux Pulse'):
#                     exp.play(signal='flux', pulse=fast_flux_pulse)
#                 with exp.section(uid='Fast Flux Pulse',):
#                     exp.play(signal='drive', pulse=pulse_library.gaussian_square(
#                         uid=f"drive_spec_pulse_{qubit.uid}",
#                         length=qubit.parameters.user_defined["pulse_length"],
#                         amplitude=1))
#                 with exp.section(uid='readout', play_after='Fast Flux Pulse'):
#                     exp.play(signal="measure", pulse=readout_pulse)
#                     exp.acquire(signal='acquire', handle='single_freq_data', length=qubit.parameters.user_defined['readout_len'], kernel=readout_pulse)
#             with exp.section(uid="delay_between_readout", length=10e-6):
#                 exp.reserve('measure')
#                 exp.reserve('acquire')
#                 exp.reserve('drive')
#                 exp.reserve('flux')
#     exp_calibration = Calibration()
#     readout_osc = Oscillator(
#         "readout_osc",
#         frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
#         modulation_type=ModulationType.SOFTWARE)
#     flux_osc = Oscillator(
#         "flux_osc",
#         frequency=0,
#         modulation_type=ModulationType.AUTO,)
#     flux_lo = Oscillator(
#         "flux_lo",
#         frequency=0,)
#     drive_osc = Oscillator(
#         "drive_osc",
#         frequency=drive_freq_sweep,
#         modulation_type=ModulationType.HARDWARE)
#     drive_lo = Oscillator(
#         'drive_lo',
#         frequency=qubit.parameters.drive_lo_frequency,
#         modulation_type=ModulationType.HARDWARE)
#     exp_calibration["measure"] = SignalCalibration(
#         oscillator=readout_osc)
#     exp_calibration["drive"] = SignalCalibration(
#         oscillator=drive_osc,
#         local_oscillator=drive_lo,)
#     exp_calibration["flux"] = SignalCalibration(
#         oscillator=flux_osc,
#         local_oscillator=flux_lo,
#         port_mode=PortMode.LF)
#     exp.set_calibration(exp_calibration)
#     return exp
    
# exp = fast_flux_pulse_exp()

drive_freq_sweep = LinearSweepParameter(
    uid='Drive_Frequency',
    start=4e9-qubit.parameters.drive_lo_frequency-500e6,
    stop=4e9-qubit.parameters.drive_lo_frequency+500e6,
    count=501
)

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=2e-6,
    amplitude=qubit.parameters.user_defined['readout_amp'],
    width=qubit.parameters.user_defined['readout_len']*0.9,
    sigma=0.2,)

readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=2e-6, amplitude=1.0)
    
exp = Experiment(
    uid="Spectroscopy",
    signals=[
        ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
        ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),
        ExperimentSignal(uid="drive", map_to=lsg[qubit.uid]['drive_line']),
        ExperimentSignal(uid="flux", map_to=lsg[qubit.uid]['flux_line'])
    ]
)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.INTEGRATION,
):
    with exp.sweep(
        uid='Drive Frequency Sweep',
        parameter=drive_freq_sweep,
    ):
        with exp.section(uid='Play flux'):
            exp.play(signal='flux', pulse=fast_flux_pulse)
        with exp.section(uid='Play drive'):
            exp.play(
                signal="drive",
                pulse=pulse_library.gaussian_square(
                    uid=f"drive_spec_pulse_{qubit.uid}",
                    length=qubit.parameters.user_defined["pulse_length"],
                    amplitude=1), #qubit.parameters.user_defined['amplitude_pi']), #max power to start
            )
        with exp.section(
            uid="single_res_point_readout",
            play_after='Play drive',
        ):
            exp.reserve('drive')
            exp.play(
                signal="measure",
                pulse=readout_pulse,)
            exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
                signal='acquire',
                handle='single_freq_data',
                length=2e-6,
                kernel=readout_weighting_function #this is important to actually integrate against so the result is non-zero
            )
        with exp.section(uid="delay_between_readout",
                         play_after="single_res_point_readout",
                         length=20e-6): #qubit.parameters.user_defined['cw_reset_length']):
            exp.reserve(signal="measure")
            exp.reserve(signal="acquire")
            exp.reserve(signal="drive")


exp_calibration = Calibration()
flux_osc = Oscillator(
    "flux_osc",
    frequency=0,
    modulation_type=ModulationType.AUTO,)
flux_lo = Oscillator(
    "flux_lo",
    frequency=0,)
readout_osc = Oscillator(
    "readout_osc",
    frequency=(res_to_current(coil.current()) - qubit.parameters.readout_lo_frequency),
    modulation_type=ModulationType.SOFTWARE)
drive_osc = Oscillator(
    "drive_osc",
    frequency=drive_freq_sweep,
    modulation_type=ModulationType.HARDWARE, )
drive_lo_lf = Oscillator(
    "drive_lo_lf",
    frequency=qubit.parameters.drive_lo_frequency)
exp_calibration["drive"] = SignalCalibration(
    oscillator=drive_osc,
    port_mode=PortMode.RF,
    local_oscillator=drive_lo_lf)
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration["flux"] = SignalCalibration(
    oscillator=flux_osc,
    local_oscillator=flux_lo,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=100,)